# Figure 2 and Supplementary Figures 9–11: spatial neighborhood analysis

This notebook reads the annotated cell dataset produced by the preceding publication notebook. Nearest neighbors are restricted to each tissue region and exclude the index cell. It reproduces the recurrent-neighborhood analyses in Supplementary Figures 9–11 and the paired immune-response comparisons in Figure 2f–i and k–n. `S2_reg004` is used only as the OVA-expression reference region. Manuscript plots display inline and are not written as image files.


## 1. Environment, parameters, and annotated input


In [ ]:
from __future__ import annotations
from pathlib import Path
import warnings
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from scipy import stats
sns.set_theme(style="white", context="notebook")
plt.rcParams.update({"figure.dpi": 150, "axes.titlesize": 12})


In [ ]:
def find_nanostamp_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "Manuscripts" / "NanoSTAMP"
        if candidate.exists():
            return candidate
        if base.name == "NanoSTAMP" and (base / "Data").exists():
            return base
    raise FileNotFoundError(
        "Could not locate Manuscripts/NanoSTAMP. Start Jupyter inside the repository "
        "or set NANOSTAMP_ROOT to the publication package directory."
    )


NANOSTAMP_ROOT = find_nanostamp_root()
DATA_ROOT = NANOSTAMP_ROOT / "Data" / "Figure_2_and_Supplementary_Figures_6_11_Multiplex_LNP"
INPUT_ROOT = DATA_ROOT / "Precomputed_Downstream_Input"
GENERATED_ROOT = DATA_ROOT / "Generated_Output"

from IPython.display import display

RANDOM_STATE = 0
K_NEIGHBORS = 10
N_NEIGHBORHOODS = 12
ELBOW_CLUSTER_VALUES = [5, 8, 10, 12, 15, 20, 25, 30]
ELBOW_MAX_CELLS = 250_000
LNP_ORDER = [f"LNP_{i:02d}" for i in range(1, 11)]
GATE_REFERENCE_REGIONS = ["S2_reg004"]
GATE_QUANTILE = 0.95

CELL_OUTPUT_ROOT = GENERATED_ROOT / "Cell_and_Functional_Analysis"
CELL_TABLE_DIR = CELL_OUTPUT_ROOT / "Tables"
INPUT_H5AD = CELL_OUTPUT_ROOT / "Figure_2_annotated_cells.h5ad"
NEIGHBOR_OUTPUT_ROOT = GENERATED_ROOT / "Spatial_Neighborhood_Analysis"
NEIGHBOR_TABLE_DIR = NEIGHBOR_OUTPUT_ROOT / "Tables"
TABLE_DIR = NEIGHBOR_TABLE_DIR
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"{INPUT_H5AD} was not found. Run the cleaned cell and functional analysis notebook first."
    )

adata = ad.read_h5ad(INPUT_H5AD)
print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} markers from {INPUT_H5AD}")


In [ ]:
LNP_REGIONS = [f'S{slide}_reg{region:03d}' for slide in range(1, 5) for region in range(4)]
required = {'x', 'y', 'lnp_region', 'condition', 'cell_type', 'lnp_call', 'lnp_positive', 'Fluc', 'OVA', 'SIINFEKL_H-2Kb', 'CD86'}
missing = sorted(required - set(adata.obs.columns))
if missing:
    raise KeyError(f'Final H5AD is missing required columns: {missing}')
cells = adata.obs.copy()
cells['x'] = pd.to_numeric(cells['x'], errors='coerce')
cells['y'] = pd.to_numeric(cells['y'], errors='coerce')
cells['Fluc'] = pd.to_numeric(cells['Fluc'], errors='coerce')
cells['OVA'] = pd.to_numeric(cells['OVA'], errors='coerce')
cells['SIINFEKL_H-2Kb'] = pd.to_numeric(cells['SIINFEKL_H-2Kb'], errors='coerce')
cells['CD86'] = pd.to_numeric(cells['CD86'], errors='coerce')
cells['cell_type'] = cells['cell_type'].astype('string').str.strip().fillna('Unknown')
cells['lnp_call'] = cells['lnp_call'].astype('string').fillna('no_barcode')
cells['lnp_positive'] = cells['lnp_positive'].fillna(False).astype(bool)
cells = cells.loc[cells['lnp_region'].isin([*LNP_REGIONS, *GATE_REFERENCE_REGIONS])].dropna(subset=['x', 'y', 'lnp_region']).copy()
gate_rows = []
for (display_name, marker) in {'Luc': 'Fluc', 'OVA': 'OVA'}.items():
    reference = cells.loc[cells['lnp_region'].isin(GATE_REFERENCE_REGIONS), marker].dropna()
    if reference.empty:
        raise ValueError(f'No values available to gate {marker} in {GATE_REFERENCE_REGIONS}')
    threshold = float(reference.quantile(GATE_QUANTILE))
    cells[f'{display_name.lower()}_positive'] = cells[marker] > threshold
    gate_rows.append({'marker': display_name, 'column': marker, 'threshold': threshold, 'quantile': GATE_QUANTILE, 'reference_regions': ','.join(GATE_REFERENCE_REGIONS), 'n_reference_cells': int(reference.size)})
gate_table = pd.DataFrame(gate_rows)
gate_table.to_csv(TABLE_DIR / 'luc_ova_neighborhood_gate_thresholds.csv', index=False)
display(gate_table)
display(cells.groupby(['condition', 'lnp_region'], observed=True).size().rename('n_cells').reset_index())


## 2. Build region-restricted 10-nearest-neighbor windows


In [ ]:
def pool_cell_types(series: pd.Series) -> pd.Series:
    values = series.astype('string').fillna('Unknown')
    return values.mask(values.str.contains('epithelial', case=False, na=False), 'Epithelial')

def build_neighbor_composition(df, k=10, region_col='lnp_region', type_col='cell_type_pooled'):
    work = df.reset_index(names='source_obs_name').reset_index(names='cell_row')
    categories = sorted(work[type_col].dropna().astype(str).unique())
    type_to_idx = {name: i for (i, name) in enumerate(categories)}
    one_hot = np.zeros((len(work), len(categories)), dtype=np.float32)
    one_hot[np.arange(len(work)), work[type_col].astype(str).map(type_to_idx).to_numpy()] = 1
    counts = np.zeros_like(one_hot)
    distances = np.full(len(work), np.nan, dtype=np.float32)
    for (region, idx) in work.groupby(region_col, observed=True).groups.items():
        idx = np.asarray(list(idx), dtype=int)
        n_available = len(idx) - 1
        if n_available < 1:
            warnings.warn(f'Skipping {region}: fewer than two cells')
            continue
        k_here = min(k, n_available)
        xy = work.loc[idx, ['x', 'y']].to_numpy(float)
        nn = NearestNeighbors(n_neighbors=k_here + 1).fit(xy)
        (dist, nbr_local) = nn.kneighbors(xy)
        nbr_global = idx[nbr_local[:, 1:]]
        counts[idx] = one_hot[nbr_global].sum(axis=1)
        distances[idx] = dist[:, 1:].mean(axis=1)
    composition = pd.DataFrame(counts, columns=categories, index=work.index)
    composition = composition.div(composition.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    work['mean_neighbor_distance'] = distances
    return (work, composition)
cells['cell_type_pooled'] = pool_cell_types(cells['cell_type'])
(neighbor_cells, neighbor_composition) = build_neighbor_composition(cells, k=K_NEIGHBORS)
print(f"Built {K_NEIGHBORS}-nearest-neighbor windows for {len(neighbor_cells):,} cells across {neighbor_cells['lnp_region'].nunique()} regions")


## 3. Choose the neighborhood count with an elbow analysis


In [ ]:
valid = neighbor_composition.sum(axis=1).gt(0)
valid_index = neighbor_composition.index[valid].to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
if len(valid_index) > ELBOW_MAX_CELLS:
    elbow_index = np.sort(rng.choice(valid_index, size=ELBOW_MAX_CELLS, replace=False))
else:
    elbow_index = valid_index
elbow_matrix = neighbor_composition.loc[elbow_index].to_numpy(dtype=np.float32)
candidate_clusters = [k for k in ELBOW_CLUSTER_VALUES if 1 < k < len(elbow_matrix)]
elbow_rows = []
for n_clusters_candidate in candidate_clusters:
    elbow_model = MiniBatchKMeans(n_clusters=n_clusters_candidate, random_state=RANDOM_STATE, n_init=10, batch_size=4096)
    elbow_model.fit(elbow_matrix)
    elbow_rows.append({'n_clusters': n_clusters_candidate, 'inertia': elbow_model.inertia_, 'inertia_per_sample': elbow_model.inertia_ / len(elbow_matrix)})
    print(f'Finished elbow fit for {n_clusters_candidate} clusters')
elbow_results = pd.DataFrame(elbow_rows)
elbow_results['relative_inertia_reduction'] = -elbow_results['inertia_per_sample'].diff() / elbow_results['inertia_per_sample'].shift(1)
elbow_results.to_csv(TABLE_DIR / 'neighborhood_kmeans_elbow_results.csv', index=False)
(fig, axes) = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(elbow_results['n_clusters'], elbow_results['inertia_per_sample'], marker='o')
axes[0].axvline(N_NEIGHBORHOODS, color='#D55E00', linestyle='--', label=f'Current choice = {N_NEIGHBORHOODS}')
axes[0].set(xlabel='Number of K-means clusters', ylabel='Inertia per sampled cell', title='Neighborhood-cluster elbow curve')
axes[0].legend(frameon=False)
axes[1].plot(elbow_results['n_clusters'], 100 * elbow_results['relative_inertia_reduction'], marker='o')
axes[1].axvline(N_NEIGHBORHOODS, color='#D55E00', linestyle='--')
axes[1].set(xlabel='Number of K-means clusters', ylabel='Reduction from previous candidate (%)', title='Incremental improvement')
sns.despine()
plt.tight_layout()
plt.show()
display(elbow_results)


## 4. Fit and manually name recurrent neighborhoods


In [ ]:
n_clusters = min(N_NEIGHBORHOODS, int(valid.sum()))
model = MiniBatchKMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=20, batch_size=4096)
neighbor_cells['neighborhood_id'] = pd.NA
neighbor_cells.loc[valid, 'neighborhood_id'] = model.fit_predict(neighbor_composition.loc[valid]).astype(int)
centroids = pd.DataFrame(model.cluster_centers_, columns=neighbor_composition.columns)
centroids.index.name = 'neighborhood_id'
top2 = centroids.apply(lambda row: row.nlargest(min(2, len(row))).index.tolist(), axis=1)
neighborhood_name_map = {i: f"N{i + 1:02d}: {' + '.join(names)}-rich" for (i, names) in top2.items()}
neighbor_cells['neighborhood'] = neighbor_cells['neighborhood_id'].map(neighborhood_name_map)
neighborhood_sizes = neighbor_cells.groupby('neighborhood', observed=True).size().sort_values(ascending=False)
ordered_names = neighborhood_sizes.index.tolist()
centroid_plot = centroids.rename(index=neighborhood_name_map).reindex(ordered_names) * 100
centroid_plot.to_csv(TABLE_DIR / 'neighborhood_cell_type_composition_percent.csv')
neighbor_cells.to_csv(TABLE_DIR / 'cell_neighborhood_assignments.csv', index=False)
(fig, ax) = plt.subplots(figsize=(max(9, 0.42 * centroid_plot.shape[1]), max(5, 0.42 * centroid_plot.shape[0])))
sns.heatmap(centroid_plot, cmap='Reds', vmin=0, vmax=min(60, np.ceil(centroid_plot.to_numpy().max() / 10) * 10), cbar_kws={'label': f'Mean cell-type fraction among {K_NEIGHBORS} neighbors (%)'}, ax=ax)
ax.set(title='Recurrent spatial neighborhood composition', xlabel='Neighbor cell type', ylabel='Neighborhood')
plt.tight_layout()
plt.show()
display(neighborhood_sizes.rename('n_index_cells').to_frame())


### Merge manually annotated neighborhood classes


In [ ]:
MANUAL_NEIGHBORHOOD_NAMES = {0: 'CD8+ T cell enriched', 1: 'B cell enriched', 2: 'Macrophage enriched', 3: 'Endothelial enriched', 4: 'Fibroblast enriched', 5: 'CD4+ T cell enriched', 6: 'DC enriched', 7: 'B cell and Endothelial enriched', 8: 'Macrophage and Endothelial enriched', 9: 'Muscle + Macrophage enriched', 10: 'B cell enriched', 11: 'Stromal + Neutrophil enriched'}
unknown_ids = sorted(set(MANUAL_NEIGHBORHOOD_NAMES) - set(centroids.index))
if unknown_ids:
    raise KeyError(f'Unknown neighborhood IDs: {unknown_ids}')
manual_neighborhood_name_map = {}
for neighborhood_id in centroids.index:
    custom_name = str(MANUAL_NEIGHBORHOOD_NAMES.get(neighborhood_id, '')).strip()
    automatic_name = ' + '.join(top2.loc[neighborhood_id]) + '-rich'
    manual_neighborhood_name_map[neighborhood_id] = custom_name if custom_name else automatic_name
neighborhood_name_map = manual_neighborhood_name_map
neighbor_cells['neighborhood'] = neighbor_cells['neighborhood_id'].map(neighborhood_name_map)
neighborhood_sizes = neighbor_cells.groupby('neighborhood', observed=True).size().sort_values(ascending=False)
ordered_names = neighborhood_sizes.index.tolist()
cluster_sizes_by_id = neighbor_cells['neighborhood_id'].value_counts().reindex(centroids.index, fill_value=0).astype(float)
weighted_centroids = centroids.mul(cluster_sizes_by_id, axis=0)
weighted_centroids['neighborhood'] = [neighborhood_name_map[i] for i in centroids.index]
merged_centroid_sums = weighted_centroids.groupby('neighborhood', sort=False).sum()
merged_weights = pd.Series(cluster_sizes_by_id.to_numpy(), index=[neighborhood_name_map[i] for i in centroids.index]).groupby(level=0, sort=False).sum()
centroid_plot = merged_centroid_sums.div(merged_weights, axis=0).reindex(ordered_names) * 100
name_table = pd.DataFrame({'source_cluster_id': list(centroids.index), 'merged_neighborhood_name': [neighborhood_name_map[i] for i in centroids.index], 'n_index_cells': [int((neighbor_cells['neighborhood_id'] == i).sum()) for i in centroids.index]})
name_table.to_csv(TABLE_DIR / 'neighborhood_name_key.csv', index=False)
centroid_plot.to_csv(TABLE_DIR / 'neighborhood_cell_type_composition_percent.csv')
neighbor_cells.to_csv(TABLE_DIR / 'cell_neighborhood_assignments.csv', index=False)
(fig, ax) = plt.subplots(figsize=(max(9, 0.42 * centroid_plot.shape[1]), max(5, 0.42 * centroid_plot.shape[0])))
heatmap_max = min(60, max(10, np.ceil(centroid_plot.to_numpy().max() / 10) * 10))
sns.heatmap(centroid_plot, cmap='Reds', vmin=0, vmax=heatmap_max, cbar_kws={'label': f'Mean cell-type fraction among {K_NEIGHBORS} neighbors (%)'}, ax=ax)
ax.set(title='Manually named spatial neighborhoods', xlabel='Neighbor cell type', ylabel='Neighborhood')
plt.tight_layout()
plt.show()
display(name_table)


## 5. Neighborhood distributions of LNP-positive cells


In [ ]:
lnp_cells = neighbor_cells.loc[neighbor_cells['lnp_positive'] & neighbor_cells['lnp_call'].isin(LNP_ORDER)].copy()
lnp_neighborhood_counts = lnp_cells.groupby(['lnp_call', 'neighborhood'], observed=True).size().rename('n_cells').reset_index()
lnp_totals = lnp_cells.groupby('lnp_call', observed=True).size().rename('n_lnp_cells')
lnp_neighborhood_counts = lnp_neighborhood_counts.merge(lnp_totals, on='lnp_call')
lnp_neighborhood_counts['percent'] = 100 * lnp_neighborhood_counts['n_cells'] / lnp_neighborhood_counts['n_lnp_cells']
lnp_neighborhood_counts.to_csv(TABLE_DIR / 'corrected_lnp_neighborhood_abundance.csv', index=False)
lnp_plot = lnp_neighborhood_counts.pivot(index='lnp_call', columns='neighborhood', values='percent').fillna(0)
lnp_plot = lnp_plot.reindex(index=LNP_ORDER, columns=ordered_names)
(fig, ax) = plt.subplots(figsize=(max(10, 0.65 * len(ordered_names)), 6))
sns.heatmap(lnp_plot, cmap='Purples', annot=True, fmt='.1f', cbar_kws={'label': '% of decoded LNP+ cells'}, ax=ax)
ax.set(title='Spatial neighborhood distribution by corrected LNP identity', xlabel='Neighborhood', ylabel='Corrected LNP call')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()
pie_palette = dict(zip(ordered_names, sns.color_palette('tab20', n_colors=len(ordered_names))))
(fig, axes) = plt.subplots(2, 5, figsize=(20, 8.2), subplot_kw={'aspect': 'equal'})
for (ax, lnp_name) in zip(axes.flat, LNP_ORDER):
    values = lnp_plot.loc[lnp_name, ordered_names].fillna(0).to_numpy(float)
    total = values.sum()
    if total <= 0:
        ax.text(0.5, 0.5, 'No decoded LNP+ cells', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(lnp_name.replace('_', ' '), fontsize=12, fontweight='bold')
        ax.axis('off')
        continue
    ax.pie(values, colors=[pie_palette[name] for name in ordered_names], startangle=90, counterclock=False, autopct=lambda pct: f'{pct:.1f}%' if pct >= 4 else '', pctdistance=0.72, textprops={'fontsize': 8}, wedgeprops={'linewidth': 0.6, 'edgecolor': 'white'})
    n_lnp_cells = int(lnp_totals.get(lnp_name, 0))
    ax.set_title(f"{lnp_name.replace('_', ' ')}\n(n={n_lnp_cells:,})", fontsize=12, fontweight='bold')
legend_handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=8, markerfacecolor=pie_palette[name], markeredgecolor='none', label=name) for name in ordered_names]
fig.legend(handles=legend_handles, labels=ordered_names, title='Neighborhood', loc='center left', bbox_to_anchor=(0.82, 0.5), frameon=False, fontsize=9, title_fontsize=10)
fig.suptitle('Spatial neighborhood distribution by corrected LNP identity', fontsize=17, y=0.98)
fig.subplots_adjust(left=0.03, right=0.8, bottom=0.04, top=0.9, wspace=0.08, hspace=0.24)
plt.show()


## 6. Local cell-type enrichment around LNP-positive cells


In [ ]:
composition_with_meta = pd.concat([neighbor_cells[['lnp_region', 'condition', 'lnp_positive', 'lnp_call', 'luc_positive', 'ova_positive']], neighbor_composition.add_prefix('neighbor__')], axis=1)
region_composition = composition_with_meta.groupby(['lnp_region', 'condition', 'lnp_positive'], observed=True).mean(numeric_only=True).reset_index()
region_composition.to_csv(TABLE_DIR / 'mean_neighbor_composition_by_region_and_lnp_status.csv', index=False)
neighbor_cols = [c for c in region_composition if c.startswith('neighbor__')]
wide = region_composition.pivot(index='lnp_region', columns='lnp_positive', values=neighbor_cols)
enrichment_rows = []
for region in wide.index:
    for col in neighbor_cols:
        neg = wide.loc[region, (col, False)] if (col, False) in wide.columns else np.nan
        pos = wide.loc[region, (col, True)] if (col, True) in wide.columns else np.nan
        enrichment_rows.append({'lnp_region': region, 'cell_type': col.removeprefix('neighbor__'), 'lnp_negative_fraction': neg, 'lnp_positive_fraction': pos, 'log2_enrichment_lnp_pos_vs_neg': np.log2((pos + 0.0001) / (neg + 0.0001))})
region_enrichment = pd.DataFrame(enrichment_rows)
region_enrichment.to_csv(TABLE_DIR / 'neighbor_celltype_enrichment_around_lnp_positive_by_region.csv', index=False)
enrichment_summary = region_enrichment.groupby('cell_type', observed=True)['log2_enrichment_lnp_pos_vs_neg'].agg(['count', 'mean', 'sem']).sort_values('mean')
(fig, ax) = plt.subplots(figsize=(7, max(5, 0.3 * len(enrichment_summary))))
ax.barh(enrichment_summary.index, enrichment_summary['mean'], xerr=enrichment_summary['sem'], color=np.where(enrichment_summary['mean'] >= 0, '#D55E00', '#0072B2'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set(xlabel='Mean within-region log2 enrichment around LNP+ vs LNP− cells', ylabel='Neighbor cell type', title='Cell types locally enriched around decoded LNP+ cells')
plt.tight_layout()
plt.show()
display(enrichment_summary)


## 7. Shared focused-neighborhood and paired-test helpers


In [ ]:
from scipy import stats

def summarize_focused_neighbors(target_df, group_col, group_order, file_prefix):
    target_df = target_df.copy()
    target_df[group_col] = pd.Categorical(target_df[group_col], categories=group_order, ordered=True)
    target_df = target_df[target_df[group_col].notna()].copy()
    pooled = target_df.groupby([group_col, 'neighborhood'], observed=True).size().rename('n_target_cells').reset_index()
    pooled_totals = target_df.groupby(group_col, observed=True).size().rename('n_group_cells').reset_index()
    pooled = pooled.merge(pooled_totals, on=group_col, how='left')
    pooled['percent'] = 100 * pooled['n_target_cells'] / pooled['n_group_cells']
    region = target_df.groupby(['lnp_region', group_col, 'neighborhood'], observed=True).size().rename('n_target_cells').reset_index()
    region_totals = target_df.groupby(['lnp_region', group_col], observed=True).size().rename('n_group_cells').reset_index()
    region = region.merge(region_totals, on=['lnp_region', group_col], how='left')
    region['percent'] = 100 * region['n_target_cells'] / region['n_group_cells']
    direct = pd.concat([target_df[['lnp_region', group_col]].reset_index(drop=True), neighbor_composition.loc[target_df.index].reset_index(drop=True)], axis=1)
    direct_region = direct.groupby(['lnp_region', group_col], observed=True).mean(numeric_only=True).reset_index()
    direct_pooled = direct.groupby(group_col, observed=True).mean(numeric_only=True) * 100
    pooled.to_csv(TABLE_DIR / f'{file_prefix}_merged_neighborhood_abundance.csv', index=False)
    region.to_csv(TABLE_DIR / f'{file_prefix}_merged_neighborhood_abundance_by_region.csv', index=False)
    direct_region.to_csv(TABLE_DIR / f'{file_prefix}_direct_neighbor_composition_by_region.csv', index=False)
    direct_pooled.to_csv(TABLE_DIR / f'{file_prefix}_direct_neighbor_composition_pooled_percent.csv')
    return (pooled, region, direct_pooled, direct_region)

def format_pvalue(pvalue):
    if not np.isfinite(pvalue):
        return 'P = NA'
    if pvalue < 0.0001:
        return 'P < 0.0001'
    if pvalue < 0.001:
        return f'P = {pvalue:.2e}'
    return f'P = {pvalue:.3f}'

def paired_region_ttest(region_df, group_col, groups, metric):
    wide = region_df.pivot(index='lnp_region', columns=group_col, values=metric).reindex(columns=groups).dropna()
    if len(wide) >= 2:
        result = stats.ttest_rel(wide[groups[0]], wide[groups[1]], nan_policy='omit')
        (statistic, pvalue) = (float(result.statistic), float(result.pvalue))
        degrees_of_freedom = len(wide) - 1
    else:
        (statistic, pvalue, degrees_of_freedom) = (np.nan, np.nan, np.nan)
    return {'test': 'two-sided paired Student t-test', 'metric': metric, 'group_1': groups[0], 'group_2': groups[1], 'n_paired_regions': len(wide), 'degrees_of_freedom': degrees_of_freedom, 't_statistic': statistic, 'pvalue': pvalue, f'mean_{groups[0]}': wide[groups[0]].mean() if len(wide) else np.nan, f'mean_{groups[1]}': wide[groups[1]].mean() if len(wide) else np.nan}

def connect_paired_region_dots(ax, region_df, group_col, groups, metric, region_col='lnp_region', color='#8A8A8A', linewidth=0.55, alpha=0.65):
    """Connect complete region-matched group pairs behind the plotted dots."""
    paired = region_df[[region_col, group_col, metric]].copy().assign(**{metric: lambda frame: pd.to_numeric(frame[metric], errors='coerce')}).pivot_table(index=region_col, columns=group_col, values=metric, aggfunc='mean').reindex(columns=groups).dropna()
    for (_, values) in paired.iterrows():
        ax.plot([0, 1], [values[groups[0]], values[groups[1]]], color=color, linewidth=linewidth, alpha=alpha, solid_capstyle='round', zorder=2.4)
    return paired

def annotate_paired_ttest(ax, region_df, group_col, groups, metric):
    test = paired_region_ttest(region_df, group_col, groups, metric)
    pvalue = test['pvalue']
    if not np.isfinite(pvalue):
        stars = 'NA'
    elif pvalue < 0.0001:
        stars = '****'
    elif pvalue < 0.001:
        stars = '***'
    elif pvalue < 0.01:
        stars = '**'
    elif pvalue < 0.05:
        stars = '*'
    else:
        stars = 'ns'
    (y_min, y_max) = ax.get_ylim()
    y_span = y_max - y_min if y_max > y_min else 1.0
    bracket_y = y_max + 0.04 * y_span
    bracket_h = 0.025 * y_span
    ax.plot([0, 0, 1, 1], [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y], color='#202020', linewidth=0.9, clip_on=False)
    ax.text(0.5, bracket_y + bracket_h + 0.012 * y_span, format_pvalue(pvalue), ha='center', va='bottom', fontsize=7.5)
    ax.set_ylim(y_min, y_max + 0.16 * y_span)
    return test
CD8_NEIGHBOR_K_VALUES = [3, 5, 8, 10, 15, 20, 25]
CD8_K_PLOT_ORDER = [3, 5, 8, 10, 15, 20, 25]
CD8_K_PALETTE = {k: '#9EC2D8' for k in CD8_NEIGHBOR_K_VALUES}

def add_multiscale_cd8_metrics(target_df, all_cells, ks=CD8_NEIGHBOR_K_VALUES):
    """Add CD8 presence, count, and mean CD8 distance within each KNN window."""
    out = target_df.copy()
    for k in ks:
        out[f'n_cd8_neighbors_k{k}'] = 0
        out[f'has_cd8_neighbor_k{k}'] = False
        out[f'mean_cd8_distance_k{k}'] = np.nan
    max_k = max(ks)
    for (region, query_index) in out.groupby('lnp_region', observed=True).groups.items():
        reference = all_cells[all_cells['lnp_region'].eq(region)]
        n_available = len(reference) - 1
        if n_available < 1:
            continue
        query_index = np.asarray(list(query_index))
        reference_index = reference.index.to_numpy()
        query_xy = out.loc[query_index, ['x', 'y']].to_numpy(float)
        reference_xy = reference[['x', 'y']].to_numpy(float)
        n_query_neighbors = min(max_k + 1, len(reference))
        (distance, local_index) = NearestNeighbors(n_neighbors=n_query_neighbors).fit(reference_xy).kneighbors(query_xy)
        global_index = reference_index[local_index]
        distance_without_self = np.where(global_index == query_index[:, None], np.inf, distance)
        order = np.argsort(distance_without_self, axis=1)
        ordered_distance = np.take_along_axis(distance_without_self, order, axis=1)[:, :min(max_k, n_available)]
        ordered_local = np.take_along_axis(local_index, order, axis=1)[:, :min(max_k, n_available)]
        reference_is_cd8 = reference['cell_type'].astype(str).eq('CD8+ T').to_numpy()
        ordered_is_cd8 = reference_is_cd8[ordered_local]
        for k in ks:
            k_here = min(k, n_available)
            is_cd8 = ordered_is_cd8[:, :k_here]
            cd8_distance = ordered_distance[:, :k_here]
            count = is_cd8.sum(axis=1).astype(int)
            distance_sum = np.where(is_cd8, cd8_distance, 0.0).sum(axis=1)
            mean_distance = np.divide(distance_sum, count, out=np.full(len(count), np.nan), where=count > 0)
            out.loc[query_index, f'n_cd8_neighbors_k{k}'] = count
            out.loc[query_index, f'has_cd8_neighbor_k{k}'] = count > 0
            out.loc[query_index, f'mean_cd8_distance_k{k}'] = mean_distance
    return out

def summarize_multiscale_cd8_by_region(target_df, group_col, ks=CD8_NEIGHBOR_K_VALUES):
    frames = []
    for k in ks:
        summary = target_df.groupby(['lnp_region', group_col], observed=True).agg(n_target_cells=(group_col, 'size'), n_dcs_with_cd8_neighbor=(f'has_cd8_neighbor_k{k}', 'sum'), pct_with_cd8_neighbor=(f'has_cd8_neighbor_k{k}', 'mean'), mean_n_cd8_neighbors=(f'n_cd8_neighbors_k{k}', 'mean'), mean_cd8_distance=(f'mean_cd8_distance_k{k}', 'mean')).reset_index()
        summary['pct_with_cd8_neighbor'] *= 100
        summary['k_neighbors'] = k
        frames.append(summary)
    return pd.concat(frames, ignore_index=True)
CD8_RADIUS_VALUES = [15, 20, 25, 30]
CD8_RADIUS_PLOT_ORDER = [15, 20, 25, 30]

def add_radius_cd8_metrics(target_df, all_cells, radii=CD8_RADIUS_VALUES):
    """Add CD8 presence, count, and mean distance within fixed coordinate radii."""
    out = target_df.copy()
    for radius in radii:
        out[f'n_cd8_within_r{radius}'] = 0
        out[f'has_cd8_within_r{radius}'] = False
        out[f'mean_cd8_distance_within_r{radius}'] = np.nan
    max_radius = max(radii)
    cd8_cells = all_cells[all_cells['cell_type'].astype(str).eq('CD8+ T')]
    for (region, query_index) in out.groupby('lnp_region', observed=True).groups.items():
        reference = cd8_cells[cd8_cells['lnp_region'].eq(region)]
        if reference.empty:
            continue
        query_index = np.asarray(list(query_index))
        query_xy = out.loc[query_index, ['x', 'y']].to_numpy(float)
        reference_xy = reference[['x', 'y']].to_numpy(float)
        (distances, _) = NearestNeighbors().fit(reference_xy).radius_neighbors(query_xy, radius=max_radius, return_distance=True, sort_results=True)
        for radius in radii:
            counts = np.fromiter((np.count_nonzero(d <= radius) for d in distances), dtype=int)
            means = np.fromiter((d[d <= radius].mean() if np.any(d <= radius) else np.nan for d in distances), dtype=float)
            out.loc[query_index, f'n_cd8_within_r{radius}'] = counts
            out.loc[query_index, f'has_cd8_within_r{radius}'] = counts > 0
            out.loc[query_index, f'mean_cd8_distance_within_r{radius}'] = means
    return out

def summarize_radius_cd8_by_region(target_df, group_col, radii=CD8_RADIUS_VALUES):
    frames = []
    for radius in radii:
        summary = target_df.groupby(['lnp_region', group_col], observed=True).agg(n_target_cells=(group_col, 'size'), n_dcs_with_cd8=('has_cd8_within_r%d' % radius, 'sum'), pct_with_cd8=('has_cd8_within_r%d' % radius, 'mean'), mean_n_cd8=('n_cd8_within_r%d' % radius, 'mean'), mean_cd8_distance=('mean_cd8_distance_within_r%d' % radius, 'mean')).reset_index()
        summary['pct_with_cd8'] *= 100
        summary['radius'] = radius
        frames.append(summary)
    return pd.concat(frames, ignore_index=True)
HYBRID_K = 10
HYBRID_RADIUS = 25
HYBRID_RADIUS_VALUES = [25]
HYBRID_SETTINGS = [(10, 25)]
HYBRID_PLOT_SPECS = [('pct_with_cd8_neighbor', 'DCs with ≥1 CD8+ T neighbor (%)', 'Nearby CD8+ T presence'), ('mean_n_cd8_neighbors', 'Mean CD8+ T count', 'CD8+ T count'), ('median_n_cd8_neighbors', 'Median CD8+ T count', 'Median CD8+ T count'), ('median_cd8_distance', 'Median CD8+ T distance\n(native coordinate units)', 'Median CD8+ T distance')]

def add_hybrid_knn_radius_cd8_metrics(target_df, all_cells, k=HYBRID_K, radius=HYBRID_RADIUS):
    """Use at most K nearest cells, retaining only cells within the radius."""
    out = target_df.copy()
    out['hybrid_n_retained_neighbors'] = 0
    out['hybrid_n_cd8_neighbors'] = 0
    out['hybrid_has_cd8_neighbor'] = False
    out['hybrid_mean_cd8_distance'] = np.nan
    out['hybrid_median_cd8_distance'] = np.nan
    for (region, query_index) in out.groupby('lnp_region', observed=True).groups.items():
        reference = all_cells[all_cells['lnp_region'].eq(region)]
        n_available = len(reference) - 1
        if n_available < 1:
            continue
        query_index = np.asarray(list(query_index))
        reference_index = reference.index.to_numpy()
        query_xy = out.loc[query_index, ['x', 'y']].to_numpy(float)
        reference_xy = reference[['x', 'y']].to_numpy(float)
        n_query_neighbors = min(k + 1, len(reference))
        (distance, local_index) = NearestNeighbors(n_neighbors=n_query_neighbors).fit(reference_xy).kneighbors(query_xy)
        global_index = reference_index[local_index]
        distance_no_self = np.where(global_index == query_index[:, None], np.inf, distance)
        order = np.argsort(distance_no_self, axis=1)
        k_here = min(k, n_available)
        ordered_distance = np.take_along_axis(distance_no_self, order, axis=1)[:, :k_here]
        ordered_local = np.take_along_axis(local_index, order, axis=1)[:, :k_here]
        within_radius = ordered_distance < radius
        reference_is_cd8 = reference['cell_type'].astype(str).eq('CD8+ T').to_numpy()
        is_cd8 = reference_is_cd8[ordered_local] & within_radius
        retained_count = within_radius.sum(axis=1).astype(int)
        cd8_count = is_cd8.sum(axis=1).astype(int)
        cd8_distance_sum = np.where(is_cd8, ordered_distance, 0.0).sum(axis=1)
        mean_cd8_distance = np.divide(cd8_distance_sum, cd8_count, out=np.full(len(cd8_count), np.nan), where=cd8_count > 0)
        median_cd8_distance = np.array([np.median(ordered_distance[row, is_cd8[row]]) if cd8_count[row] > 0 else np.nan for row in range(len(cd8_count))])
        out.loc[query_index, 'hybrid_n_retained_neighbors'] = retained_count
        out.loc[query_index, 'hybrid_n_cd8_neighbors'] = cd8_count
        out.loc[query_index, 'hybrid_has_cd8_neighbor'] = cd8_count > 0
        out.loc[query_index, 'hybrid_mean_cd8_distance'] = mean_cd8_distance
        out.loc[query_index, 'hybrid_median_cd8_distance'] = median_cd8_distance
    return out

def summarize_hybrid_cd8_by_region(target_df, group_col):
    summary = target_df.groupby(['lnp_region', group_col], observed=True).agg(n_target_cells=(group_col, 'size'), mean_retained_neighbors=('hybrid_n_retained_neighbors', 'mean'), n_target_dcs_with_cd8=('hybrid_has_cd8_neighbor', 'sum'), pct_with_cd8_neighbor=('hybrid_has_cd8_neighbor', 'mean'), mean_n_cd8_neighbors=('hybrid_n_cd8_neighbors', 'mean'), median_n_cd8_neighbors=('hybrid_n_cd8_neighbors', 'median'), mean_cd8_distance=('hybrid_mean_cd8_distance', 'mean'), median_cd8_distance=('hybrid_median_cd8_distance', 'median')).reset_index()
    summary['pct_with_cd8_neighbor'] *= 100
    return summary


## 8. SIINFEKL-H-2Kb-positive dendritic cells and nearby CD8+ T cells


In [ ]:
DC_LNPS = ['LNP_08', 'LNP_10']
SIINFEKL_GATE_QUANTILE = 0.9
neighbor_cells['SIINFEKL_H-2Kb'] = pd.to_numeric(neighbor_cells['SIINFEKL_H-2Kb'], errors='coerce')
neighbor_cells['CD86'] = pd.to_numeric(neighbor_cells['CD86'], errors='coerce')
siinfekl_reference = neighbor_cells.loc[neighbor_cells['lnp_region'].isin(GATE_REFERENCE_REGIONS), 'SIINFEKL_H-2Kb'].dropna()
SIINFEKL_THRESHOLD = float(siinfekl_reference.quantile(SIINFEKL_GATE_QUANTILE))
neighbor_cells['siinfekl_positive'] = neighbor_cells['SIINFEKL_H-2Kb'] > SIINFEKL_THRESHOLD
dc_targets = neighbor_cells.loc[neighbor_cells['cell_type'].astype(str).eq('DC') & neighbor_cells['lnp_positive'] & neighbor_cells['lnp_call'].isin(DC_LNPS) & neighbor_cells['siinfekl_positive']].copy()
dc_targets['lnp_call'] = dc_targets['lnp_call'].astype(str)
cd8_col = 'CD8+ T'
dc_targets['cd8_neighbor_fraction'] = neighbor_composition.loc[dc_targets.index, cd8_col] if cd8_col in neighbor_composition else 0.0
dc_targets['n_cd8_neighbors'] = np.rint(dc_targets['cd8_neighbor_fraction'] * K_NEIGHBORS).astype(int)
dc_targets['has_cd8_neighbor'] = dc_targets['n_cd8_neighbors'] > 0
dc_targets['nearest_cd8_distance'] = np.nan
dc_cd8_reference = neighbor_cells[neighbor_cells['cell_type'].astype(str).eq('CD8+ T')]
for (region, query_index) in dc_targets.groupby('lnp_region', observed=True).groups.items():
    region_cd8 = dc_cd8_reference[dc_cd8_reference['lnp_region'].eq(region)]
    if region_cd8.empty:
        continue
    query_xy = dc_targets.loc[query_index, ['x', 'y']].to_numpy(float)
    reference_xy = region_cd8[['x', 'y']].to_numpy(float)
    (distance, _) = NearestNeighbors(n_neighbors=1).fit(reference_xy).kneighbors(query_xy)
    dc_targets.loc[query_index, 'nearest_cd8_distance'] = distance[:, 0]
dc_targets = add_multiscale_cd8_metrics(dc_targets, neighbor_cells)
dc_multiscale_region = summarize_multiscale_cd8_by_region(dc_targets, 'lnp_call')
dc_multiscale_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_multiscale_by_region.csv', index=False)
multiscale_cols = [c for c in dc_targets.columns if c.startswith(('n_cd8_neighbors_k', 'has_cd8_neighbor_k', 'mean_cd8_distance_k'))]
dc_targets[['source_obs_name', 'lnp_region', 'lnp_call'] + multiscale_cols].to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_multiscale_per_cell.csv', index=False)
dc_targets = add_radius_cd8_metrics(dc_targets, neighbor_cells)
dc_radius_region = summarize_radius_cd8_by_region(dc_targets, 'lnp_call')
dc_radius_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_radius_by_region.csv', index=False)
dc_hybrid_frames = []
dc_hybrid_test_frames = []
for (hybrid_k, hybrid_radius) in HYBRID_SETTINGS:
    dc_hybrid_targets = add_hybrid_knn_radius_cd8_metrics(dc_targets, neighbor_cells, k=hybrid_k, radius=hybrid_radius)
    hybrid_summary = summarize_hybrid_cd8_by_region(dc_hybrid_targets, 'lnp_call')
    hybrid_summary['radius'] = hybrid_radius
    hybrid_summary['k_neighbors'] = hybrid_k
    dc_hybrid_frames.append(hybrid_summary)
    hybrid_tests = pd.DataFrame([paired_region_ttest(hybrid_summary, 'lnp_call', DC_LNPS, metric) for metric in ['pct_with_cd8_neighbor', 'mean_n_cd8_neighbors', 'median_n_cd8_neighbors', 'median_cd8_distance']])
    hybrid_tests['radius'] = hybrid_radius
    hybrid_tests['k_neighbors'] = hybrid_k
    dc_hybrid_test_frames.append(hybrid_tests)
dc_hybrid_region = pd.concat(dc_hybrid_frames, ignore_index=True)
dc_hybrid_paired_tests = pd.concat(dc_hybrid_test_frames, ignore_index=True)
dc_hybrid_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_hybrid_settings_by_region.csv', index=False)
dc_hybrid_paired_tests.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_hybrid_settings_paired_ttests.csv', index=False)
(dc_pooled, dc_region_neighborhood, dc_direct_pooled, dc_direct_region) = summarize_focused_neighbors(dc_targets, 'lnp_call', DC_LNPS, 'lnp08_vs_lnp10_siinfekl_positive_dc')
dc_region_metrics = dc_targets.groupby(['lnp_region', 'lnp_call'], observed=True).agg(n_target_dc=('lnp_call', 'size'), mean_cd86=('CD86', 'mean'), mean_cd8_neighbor_fraction=('cd8_neighbor_fraction', 'mean'), mean_n_cd8_neighbors=('n_cd8_neighbors', 'mean'), n_dcs_with_cd8_neighbor=('has_cd8_neighbor', 'sum'), pct_with_cd8_neighbor=('has_cd8_neighbor', 'mean'), mean_nearest_cd8_distance=('nearest_cd8_distance', 'mean'), median_nearest_cd8_distance=('nearest_cd8_distance', 'median')).reset_index()
dc_region_metrics['pct_with_cd8_neighbor'] *= 100
dc_region_metrics['mean_cd8_neighbor_percent'] = 100 * dc_region_metrics['mean_cd8_neighbor_fraction']
dc_region_metrics.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_metrics_by_region.csv', index=False)
dc_targets[['source_obs_name', 'lnp_region', 'lnp_call', 'neighborhood', 'n_cd8_neighbors', 'has_cd8_neighbor', 'nearest_cd8_distance']].to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_metrics_per_cell.csv', index=False)
dc_tests = pd.DataFrame([paired_region_ttest(dc_region_metrics, 'lnp_call', DC_LNPS, metric) for metric in ['mean_cd8_neighbor_percent', 'n_dcs_with_cd8_neighbor', 'mean_nearest_cd8_distance', 'mean_cd86']])
dc_tests.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_paired_ttests.csv', index=False)
dc_niche_plot = dc_pooled.pivot(index='lnp_call', columns='neighborhood', values='percent').fillna(0)
dc_niche_plot = dc_niche_plot.reindex(index=DC_LNPS, columns=ordered_names, fill_value=0)
(fig, ax) = plt.subplots(figsize=(4.8, max(4.2, 0.42 * len(ordered_names))))
sns.heatmap(dc_niche_plot.T, cmap='Blues', annot=True, fmt='.1f', linewidths=0.5, linecolor='white', cbar_kws={'label': '% of SIINFEKL+ LNP+ DCs', 'shrink': 0.72}, ax=ax)
ax.set(title='Neighborhoods of SIINFEKL-presenting DCs', xlabel='', ylabel='')
ax.set_xticklabels(['LNP 8', 'LNP 10'], rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.show()
(hybrid_k, hybrid_radius) = (10, 25)
plot_data = dc_hybrid_region[dc_hybrid_region['radius'].eq(hybrid_radius) & dc_hybrid_region['k_neighbors'].eq(hybrid_k)].copy()
if set(plot_data['lnp_call'].dropna().astype(str)) != set(DC_LNPS):
    raise RuntimeError(f'Hybrid summary does not contain both {DC_LNPS} at K={hybrid_k}, radius={hybrid_radius} pixels. Re-run the helper/settings cell and this entire analysis cell to rebuild the summary.')
for (metric, ylabel, title) in HYBRID_PLOT_SPECS:
    (fig, ax) = plt.subplots(figsize=(5.4, 5.2))
    sns.boxplot(data=plot_data, x='lnp_call', y=metric, order=DC_LNPS, color='#9EC2D8', width=0.26, showfliers=False, linewidth=1.0, ax=ax)
    sns.stripplot(data=plot_data, x='lnp_call', y=metric, order=DC_LNPS, color='#202020', size=4.2, jitter=False, ax=ax)
    ax.set(xlabel='', ylabel=ylabel)
    ax.set_title(f'{title}\nK={hybrid_k}, R<{hybrid_radius * 0.5:g} µm', fontsize=13, pad=14)
    ax.set_ylabel(ylabel, fontsize=11, labelpad=10)
    ax.tick_params(axis='both', labelsize=10)
    ax.set_xticklabels(['LNP 8', 'LNP 10'])
    ax.set_xlim(-0.55, 1.55)
    connect_paired_region_dots(ax, plot_data, 'lnp_call', DC_LNPS, metric)
    annotate_paired_ttest(ax, plot_data, 'lnp_call', DC_LNPS, metric)
    ax.grid(axis='y', color='#E6E6E6', linewidth=0.65)
    sns.despine(ax=ax)
    fig.subplots_adjust(left=0.2, right=0.97, bottom=0.14, top=0.78)
    metric_slug = metric.replace('pct_with_', 'percent_').replace('mean_', 'mean_').replace('median_', 'median_')
    plt.show()
SECTION8A_CD8_STATE_MARKERS = {'CD3': 'CD3', 'TCRb': 'TCRβ', 'CD8a': 'CD8α', 'CD45.2': 'CD45.2', 'CD90': 'CD90', 'CD25': 'CD25', 'CD27': 'CD27', 'CD28': 'CD28', 'CD44': 'CD44', 'CD62L': 'CD62L', 'CCR7': 'CCR7', 'CD103': 'CD103', 'SCA1': 'SCA-1', 'PD1': 'PD-1', 'CD152': 'CD152 (CTLA-4)', 'GZMB': 'Granzyme B', 'FOXP3': 'FOXP3'}
section8a_missing_markers = [marker for marker in SECTION8A_CD8_STATE_MARKERS if marker not in neighbor_cells.columns]
if section8a_missing_markers:
    raise KeyError(f'Missing Section 8a CD8 marker columns: {section8a_missing_markers}')

def collect_target_cd8_neighbor_pairs(target_df, all_cells, group_col, k=10, radius=25):
    """Return target-CD8 pairs satisfying the center-excluded KNN plus strict radius rule."""
    pair_frames = []
    for (region, query_indices) in target_df.groupby('lnp_region', observed=True).groups.items():
        reference = all_cells[all_cells['lnp_region'].eq(region)]
        if len(reference) < 2:
            continue
        query_indices = np.asarray(list(query_indices))
        reference_indices = reference.index.to_numpy()
        query_xy = target_df.loc[query_indices, ['x', 'y']].to_numpy(float)
        reference_xy = reference[['x', 'y']].to_numpy(float)
        n_neighbors = min(k + 1, len(reference))
        (distances, local_indices) = NearestNeighbors(n_neighbors=n_neighbors).fit(reference_xy).kneighbors(query_xy)
        global_indices = reference_indices[local_indices]
        distances = np.where(global_indices == query_indices[:, None], np.inf, distances)
        order = np.argsort(distances, axis=1)
        k_here = min(k, len(reference) - 1)
        distances = np.take_along_axis(distances, order, axis=1)[:, :k_here]
        local_indices = np.take_along_axis(local_indices, order, axis=1)[:, :k_here]
        global_indices = reference_indices[local_indices]
        is_cd8 = reference['cell_type'].astype(str).eq('CD8+ T').to_numpy()[local_indices]
        keep = is_cd8 & (distances < radius)
        (rows, cols) = np.where(keep)
        if len(rows) == 0:
            continue
        pair_frames.append(pd.DataFrame({'lnp_region': region, group_col: target_df.loc[query_indices[rows], group_col].astype(str).to_numpy(), 'target_dc_index': query_indices[rows], 'cd8_index': global_indices[rows, cols], 'distance_pixels': distances[rows, cols]}))
    columns = ['lnp_region', group_col, 'target_dc_index', 'cd8_index', 'distance_pixels']
    return pd.concat(pair_frames, ignore_index=True) if pair_frames else pd.DataFrame(columns=columns)
dc_cd8_state_pairs = collect_target_cd8_neighbor_pairs(dc_targets, neighbor_cells, 'lnp_call', k=10, radius=25)
dc_nearby_unique_cd8 = dc_cd8_state_pairs.sort_values('distance_pixels').drop_duplicates(['lnp_region', 'lnp_call', 'cd8_index']).copy()
dc_cd8_marker_values = neighbor_cells.loc[dc_nearby_unique_cd8['cd8_index'], list(SECTION8A_CD8_STATE_MARKERS)].apply(pd.to_numeric, errors='coerce').reset_index(drop=True)
dc_nearby_unique_cd8 = pd.concat([dc_nearby_unique_cd8.reset_index(drop=True), dc_cd8_marker_values], axis=1)
dc_cd8_state_long = dc_nearby_unique_cd8.melt(id_vars=['lnp_region', 'lnp_call', 'cd8_index', 'distance_pixels'], value_vars=list(SECTION8A_CD8_STATE_MARKERS), var_name='marker', value_name='expression').dropna(subset=['expression'])
dc_cd8_state_region = dc_cd8_state_long.groupby(['lnp_region', 'lnp_call', 'marker'], observed=True).agg(n_unique_cd8=('cd8_index', 'nunique'), mean_expression=('expression', 'mean'), median_expression=('expression', 'median')).reset_index()
dc_cd8_state_pairs.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_k10_radius25_neighbor_pairs.csv', index=False)
dc_nearby_unique_cd8.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_k10_radius25_unique_cd8_state.csv', index=False)
dc_cd8_state_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_k10_radius25_state_by_region.csv', index=False)
dc_cd8_state_test_rows = []
for (marker, marker_label) in SECTION8A_CD8_STATE_MARKERS.items():
    marker_data = dc_cd8_state_region[dc_cd8_state_region['marker'].eq(marker)].copy()
    for (metric, statistic_label) in [('mean_expression', 'Mean'), ('median_expression', 'Median')]:
        test = paired_region_ttest(marker_data, 'lnp_call', DC_LNPS, metric)
        test.update({'marker': marker, 'marker_label': marker_label, 'summary_statistic': statistic_label})
        dc_cd8_state_test_rows.append(test)
        (fig, ax) = plt.subplots(figsize=(5.4, 5.2))
        sns.boxplot(data=marker_data, x='lnp_call', y=metric, order=DC_LNPS, color='#9EC2D8', width=0.26, showfliers=False, linewidth=1.0, ax=ax)
        sns.stripplot(data=marker_data, x='lnp_call', y=metric, order=DC_LNPS, color='#202020', size=4.2, jitter=False, ax=ax)
        ylabel = f'{statistic_label} {marker_label} expression'
        ax.set(xlabel='', ylabel=ylabel)
        ax.set_title(f'Nearby CD8+ T-cell {marker_label}\n{statistic_label.lower()} expression; K=10, R<12.5 µm', fontsize=13, pad=14)
        ax.set_xticklabels(['LNP 8', 'LNP 10'])
        ax.set_xlim(-0.55, 1.55)
        connect_paired_region_dots(ax, marker_data, 'lnp_call', DC_LNPS, metric)
        annotate_paired_ttest(ax, marker_data, 'lnp_call', DC_LNPS, metric)
        ax.grid(axis='y', color='#E6E6E6', linewidth=0.65)
        sns.despine(ax=ax)
        fig.subplots_adjust(left=0.2, right=0.97, bottom=0.14, top=0.78)
        plt.show()
dc_cd8_state_tests = pd.DataFrame(dc_cd8_state_test_rows)
dc_cd8_state_tests.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_k10_radius25_state_paired_ttests.csv', index=False)
display(dc_cd8_state_region)
display(dc_cd8_state_tests)
SECTION8A_DC_STATE_MARKERS = {'CD11c': 'CD11c', 'CD11b': 'CD11b', 'CD103': 'CD103', 'CD45.2': 'CD45.2', 'MHCII': 'MHC II', 'CD86': 'CD86', 'CCR7': 'CCR7', 'SIINFEKL_H-2Kb': 'SIINFEKL–H-2Kb'}
section8a_dc_missing = [m for m in SECTION8A_DC_STATE_MARKERS if m not in dc_targets.columns]
if section8a_dc_missing:
    raise KeyError(f'Missing Section 8a DC marker columns: {section8a_dc_missing}')
dc_state_values = dc_targets[['source_obs_name', 'lnp_region', 'lnp_call'] + list(SECTION8A_DC_STATE_MARKERS)].copy()
for marker in SECTION8A_DC_STATE_MARKERS:
    dc_state_values[marker] = pd.to_numeric(dc_state_values[marker], errors='coerce')
dc_state_long = dc_state_values.melt(id_vars=['source_obs_name', 'lnp_region', 'lnp_call'], value_vars=list(SECTION8A_DC_STATE_MARKERS), var_name='marker', value_name='expression').dropna(subset=['expression'])
dc_state_region = dc_state_long.groupby(['lnp_region', 'lnp_call', 'marker'], observed=True).agg(n_target_dc=('source_obs_name', 'nunique'), mean_expression=('expression', 'mean'), median_expression=('expression', 'median')).reset_index()
dc_state_values.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_state_per_cell.csv', index=False)
dc_state_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_state_by_region.csv', index=False)
dc_state_test_rows = []
for (marker, marker_label) in SECTION8A_DC_STATE_MARKERS.items():
    marker_data = dc_state_region[dc_state_region['marker'].eq(marker)].copy()
    for (metric, statistic_label) in [('mean_expression', 'Mean'), ('median_expression', 'Median')]:
        test = paired_region_ttest(marker_data, 'lnp_call', DC_LNPS, metric)
        test.update({'marker': marker, 'marker_label': marker_label, 'summary_statistic': statistic_label})
        dc_state_test_rows.append(test)
        (fig, ax) = plt.subplots(figsize=(5.4, 5.2))
        sns.boxplot(data=marker_data, x='lnp_call', y=metric, order=DC_LNPS, color='#9EC2D8', width=0.26, showfliers=False, linewidth=1.0, ax=ax)
        connect_paired_region_dots(ax, marker_data, 'lnp_call', DC_LNPS, metric)
        sns.stripplot(data=marker_data, x='lnp_call', y=metric, order=DC_LNPS, color='#202020', size=4.2, jitter=False, zorder=3, ax=ax)
        ylabel = f'{statistic_label} {marker_label} expression'
        ax.set(xlabel='', ylabel=ylabel)
        ax.set_title(f'SIINFEKL+ LNP+ DC {marker_label}\n{statistic_label.lower()} expression', fontsize=13, pad=14)
        ax.set_xticklabels(['LNP 8', 'LNP 10'])
        ax.set_xlim(-0.55, 1.55)
        annotate_paired_ttest(ax, marker_data, 'lnp_call', DC_LNPS, metric)
        ax.grid(axis='y', color='#E6E6E6', linewidth=0.65)
        sns.despine(ax=ax)
        fig.subplots_adjust(left=0.2, right=0.97, bottom=0.14, top=0.78)
        plt.show()
dc_state_tests = pd.DataFrame(dc_state_test_rows)
dc_state_tests.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_state_paired_ttests.csv', index=False)
display(dc_state_region)
display(dc_state_tests)


## 9. OVA-positive dendritic cells and nearby CD4+ T cells


In [ ]:
CD4_ACTIVATION_MARKERS = {'CD3': 'CD3', 'TCRb': 'TCRβ', 'CD4': 'CD4', 'CD45.2': 'CD45.2', 'CD90': 'CD90', 'CD25': 'CD25', 'CD27': 'CD27', 'CD28': 'CD28', 'CD44': 'CD44', 'CD62L': 'CD62L', 'CCR7': 'CCR7', 'CD103': 'CD103', 'SCA1': 'SCA-1', 'PD1': 'PD-1', 'CD152': 'CD152 (CTLA-4)', 'GZMB': 'Granzyme B', 'FOXP3': 'FOXP3'}
missing_activation_markers = [m for m in CD4_ACTIVATION_MARKERS if m not in neighbor_cells.columns]
if missing_activation_markers:
    raise KeyError(f'Missing CD4 activation-marker columns: {missing_activation_markers}')

def collect_target_cd4_neighbor_pairs(target_df, all_cells, group_col, k=10, radius=25):
    """Return target-CD4 pairs satisfying the center-excluded KNN plus radius rule."""
    pair_frames = []
    for (region, query_indices) in target_df.groupby('lnp_region', observed=True).groups.items():
        reference = all_cells[all_cells['lnp_region'].eq(region)]
        if len(reference) < 2:
            continue
        query_indices = np.asarray(list(query_indices))
        reference_indices = reference.index.to_numpy()
        query_xy = target_df.loc[query_indices, ['x', 'y']].to_numpy(float)
        reference_xy = reference[['x', 'y']].to_numpy(float)
        n_neighbors = min(k + 1, len(reference))
        (distances, local_indices) = NearestNeighbors(n_neighbors=n_neighbors).fit(reference_xy).kneighbors(query_xy)
        global_indices = reference_indices[local_indices]
        distances = np.where(global_indices == query_indices[:, None], np.inf, distances)
        order = np.argsort(distances, axis=1)
        k_here = min(k, len(reference) - 1)
        distances = np.take_along_axis(distances, order, axis=1)[:, :k_here]
        local_indices = np.take_along_axis(local_indices, order, axis=1)[:, :k_here]
        global_indices = reference_indices[local_indices]
        is_cd4 = reference['cell_type'].astype(str).eq('CD4+ T').to_numpy()[local_indices]
        keep = is_cd4 & (distances < radius)
        (rows, cols) = np.where(keep)
        if len(rows) == 0:
            continue
        pair_frames.append(pd.DataFrame({'lnp_region': region, group_col: target_df.loc[query_indices[rows], group_col].astype(str).to_numpy(), 'target_dc_index': query_indices[rows], 'cd4_index': global_indices[rows, cols], 'distance': distances[rows, cols]}))
    if not pair_frames:
        return pd.DataFrame(columns=['lnp_region', group_col, 'target_dc_index', 'cd4_index', 'distance'])
    return pd.concat(pair_frames, ignore_index=True)
ova_activation_dc_targets = neighbor_cells.loc[neighbor_cells['cell_type'].astype(str).eq('DC') & neighbor_cells['lnp_positive'] & neighbor_cells['lnp_call'].isin(DC_LNPS) & neighbor_cells['ova_positive']].copy()
ova_activation_dc_targets['lnp_call'] = ova_activation_dc_targets['lnp_call'].astype(str)
ova_dc_cd4_activation_pairs = collect_target_cd4_neighbor_pairs(ova_activation_dc_targets, neighbor_cells, 'lnp_call', k=10, radius=25)
ova_dc_cd4_target_metrics = ova_activation_dc_targets[['lnp_region', 'lnp_call']].copy()
ova_dc_cd4_target_metrics['target_dc_index'] = ova_dc_cd4_target_metrics.index
pair_metrics = ova_dc_cd4_activation_pairs.groupby('target_dc_index', observed=True).agg(n_cd4_neighbors=('cd4_index', 'nunique'), median_cd4_distance_pixels=('distance', 'median')).reset_index()
ova_dc_cd4_target_metrics = ova_dc_cd4_target_metrics.merge(pair_metrics, on='target_dc_index', how='left')
ova_dc_cd4_target_metrics['n_cd4_neighbors'] = ova_dc_cd4_target_metrics['n_cd4_neighbors'].fillna(0).astype(int)
ova_dc_cd4_target_metrics['has_cd4_neighbor'] = ova_dc_cd4_target_metrics['n_cd4_neighbors'] > 0
ova_dc_cd4_target_metrics['median_cd4_distance_um'] = ova_dc_cd4_target_metrics['median_cd4_distance_pixels'] * 0.5
ova_dc_cd4_spatial_region = ova_dc_cd4_target_metrics.groupby(['lnp_region', 'lnp_call'], observed=True).agg(n_target_dcs=('target_dc_index', 'nunique'), pct_dcs_with_cd4=('has_cd4_neighbor', 'mean'), mean_n_cd4_neighbors=('n_cd4_neighbors', 'mean'), median_n_cd4_neighbors=('n_cd4_neighbors', 'median'), median_cd4_distance_um=('median_cd4_distance_um', 'median')).reset_index()
ova_dc_cd4_spatial_region['pct_dcs_with_cd4'] *= 100
ova_dc_cd4_target_metrics.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_target_metrics.csv', index=False)
ova_dc_cd4_spatial_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_spatial_by_region.csv', index=False)
cd4_spatial_specs = [('pct_dcs_with_cd4', 'OVA+ LNP+ DCs with ≥1 CD4+ T neighbor (%)', 'Nearby CD4+ T presence'), ('mean_n_cd4_neighbors', 'Mean CD4+ T count per target DC', 'CD4+ T count'), ('median_n_cd4_neighbors', 'Median CD4+ T count per target DC', 'Median CD4+ T count'), ('median_cd4_distance_um', 'Median CD4+ T distance (µm)', 'Median CD4+ T distance')]
cd4_spatial_test_rows = []
for (metric, ylabel, title) in cd4_spatial_specs:
    test = paired_region_ttest(ova_dc_cd4_spatial_region, 'lnp_call', DC_LNPS, metric)
    cd4_spatial_test_rows.append(test)
    (fig, ax) = plt.subplots(figsize=(5.4, 5.2))
    sns.boxplot(data=ova_dc_cd4_spatial_region, x='lnp_call', y=metric, order=DC_LNPS, color='#9EC2D8', width=0.26, showfliers=False, linewidth=1.0, ax=ax)
    sns.stripplot(data=ova_dc_cd4_spatial_region, x='lnp_call', y=metric, order=DC_LNPS, color='#202020', size=4.2, jitter=False, ax=ax)
    ax.set(xlabel='', ylabel=ylabel)
    ax.set_title(f'{title}\nK=10, R<12.5 µm', fontsize=13, pad=14)
    ax.set_ylabel(ylabel, fontsize=11, labelpad=10)
    ax.tick_params(axis='both', labelsize=10)
    ax.set_xticklabels(['LNP 8', 'LNP 10'])
    ax.set_xlim(-0.55, 1.55)
    connect_paired_region_dots(ax, ova_dc_cd4_spatial_region, 'lnp_call', DC_LNPS, metric)
    annotate_paired_ttest(ax, ova_dc_cd4_spatial_region, 'lnp_call', DC_LNPS, metric)
    ax.grid(axis='y', color='#E6E6E6', linewidth=0.65)
    sns.despine(ax=ax)
    fig.subplots_adjust(left=0.2, right=0.97, bottom=0.14, top=0.78)
    plt.show()
cd4_spatial_tests = pd.DataFrame(cd4_spatial_test_rows)
cd4_spatial_tests.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_spatial_paired_ttests.csv', index=False)
nearby_cd4 = ova_dc_cd4_activation_pairs.sort_values('distance').drop_duplicates(['lnp_region', 'lnp_call', 'cd4_index']).copy()
marker_values = neighbor_cells.loc[nearby_cd4['cd4_index'], list(CD4_ACTIVATION_MARKERS)].apply(pd.to_numeric, errors='coerce').reset_index(drop=True)
nearby_cd4 = pd.concat([nearby_cd4.reset_index(drop=True), marker_values], axis=1)
activation_long = nearby_cd4.melt(id_vars=['lnp_region', 'lnp_call', 'cd4_index', 'distance'], value_vars=list(CD4_ACTIVATION_MARKERS), var_name='marker', value_name='expression').dropna(subset=['expression'])
activation_region = activation_long.groupby(['lnp_region', 'lnp_call', 'marker'], observed=True).agg(n_unique_cd4=('cd4_index', 'nunique'), mean_expression=('expression', 'mean'), median_expression=('expression', 'median')).reset_index()
ova_dc_cd4_activation_pairs.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_neighbor_pairs.csv', index=False)
nearby_cd4.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_unique_cd4_activation.csv', index=False)
activation_region.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_activation_by_region.csv', index=False)
activation_test_rows = []
for (marker, marker_label) in CD4_ACTIVATION_MARKERS.items():
    marker_data = activation_region[activation_region['marker'].eq(marker)].copy()
    for (metric, statistic_label) in [('mean_expression', 'Mean'), ('median_expression', 'Median')]:
        test = paired_region_ttest(marker_data, 'lnp_call', DC_LNPS, metric)
        test.update({'marker': marker, 'marker_label': marker_label, 'summary_statistic': statistic_label})
        activation_test_rows.append(test)
        (fig, ax) = plt.subplots(figsize=(5.4, 5.2))
        sns.boxplot(data=marker_data, x='lnp_call', y=metric, order=DC_LNPS, color='#9EC2D8', width=0.26, showfliers=False, linewidth=1.0, ax=ax)
        sns.stripplot(data=marker_data, x='lnp_call', y=metric, order=DC_LNPS, color='#202020', size=4.2, jitter=False, ax=ax)
        ylabel = f'{statistic_label} {marker_label} expression'
        ax.set(xlabel='', ylabel=ylabel)
        ax.set_title(f'Nearby CD4+ T-cell {marker_label}\n{statistic_label.lower()} expression; K=10, R<12.5 µm', fontsize=13, pad=14)
        ax.set_ylabel(ylabel, fontsize=11, labelpad=10)
        ax.tick_params(axis='both', labelsize=10)
        ax.set_xticklabels(['LNP 8', 'LNP 10'])
        ax.set_xlim(-0.55, 1.55)
        connect_paired_region_dots(ax, marker_data, 'lnp_call', DC_LNPS, metric)
        annotate_paired_ttest(ax, marker_data, 'lnp_call', DC_LNPS, metric)
        ax.grid(axis='y', color='#E6E6E6', linewidth=0.65)
        sns.despine(ax=ax)
        fig.subplots_adjust(left=0.2, right=0.97, bottom=0.14, top=0.78)
        plt.show()
activation_tests = pd.DataFrame(activation_test_rows)
activation_tests.to_csv(TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_nearby_cd4_activation_paired_ttests.csv', index=False)
display(activation_region)
display(activation_tests)


## 10. OVA-positive B-cell state and surrounding CD4+ T cells


In [ ]:
SECTION11_LNPS = ['LNP_08', 'LNP_10']
SECTION11_K = 10
SECTION11_RADIUS_PIXELS = 25
OVA_B_STATE_MARKERS = {'B220': 'B220', 'CD19': 'CD19', 'CD45.2': 'CD45.2', 'CD138': 'CD138', 'MHCII': 'MHC II', 'CD86': 'CD86', 'SIINFEKL_H-2Kb': 'SIINFEKL–H-2Kb', 'CD25': 'CD25', 'CD27': 'CD27', 'CD44': 'CD44', 'SCA1': 'SCA-1', 'PD1': 'PD-1', 'CD152': 'CD152 (CTLA-4)', 'CCR7': 'CCR7', 'CD62L': 'CD62L', 'CD11b': 'CD11b', 'CD11c': 'CD11c', 'OVA': 'OVA'}
SECTION11_CD4_STATE_MARKERS = {'CD3': 'CD3', 'TCRb': 'TCRβ', 'CD4': 'CD4', 'CD45.2': 'CD45.2', 'CD90': 'CD90', 'CD25': 'CD25', 'CD27': 'CD27', 'CD28': 'CD28', 'CD44': 'CD44', 'CD62L': 'CD62L', 'CCR7': 'CCR7', 'CD103': 'CD103', 'SCA1': 'SCA-1', 'PD1': 'PD-1', 'CD152': 'CD152 (CTLA-4)', 'GZMB': 'Granzyme B', 'FOXP3': 'FOXP3'}
section11_required = sorted(set(OVA_B_STATE_MARKERS) | set(SECTION11_CD4_STATE_MARKERS))
section11_missing = [marker for marker in section11_required if marker not in neighbor_cells.columns]
if section11_missing:
    raise KeyError(f'Missing Section 11 marker columns: {section11_missing}')

def section11_paired_figure(data, metric, ylabel, title, filename):
    """Draw the shared paired-region comparison style and save a 600-DPI figure."""
    (fig, ax) = plt.subplots(figsize=(5.4, 5.2))
    sns.boxplot(data=data, x='lnp_call', y=metric, order=SECTION11_LNPS, color='#9EC2D8', width=0.26, showfliers=False, linewidth=1.0, ax=ax)
    connect_paired_region_dots(ax, data, 'lnp_call', SECTION11_LNPS, metric)
    sns.stripplot(data=data, x='lnp_call', y=metric, order=SECTION11_LNPS, color='#202020', size=4.2, jitter=False, zorder=3, ax=ax)
    ax.set(xlabel='', ylabel=ylabel)
    ax.set_title(title, fontsize=13, pad=14)
    ax.set_xticklabels(['LNP 8', 'LNP 10'])
    ax.set_xlim(-0.55, 1.55)
    test = annotate_paired_ttest(ax, data, 'lnp_call', SECTION11_LNPS, metric)
    ax.grid(axis='y', color='#E6E6E6', linewidth=0.65)
    sns.despine(ax=ax)
    fig.subplots_adjust(left=0.2, right=0.97, bottom=0.14, top=0.78)
    plt.show()
    return test
ova_b_targets = neighbor_cells.loc[neighbor_cells['cell_type'].astype(str).eq('B') & neighbor_cells['lnp_positive'] & neighbor_cells['lnp_call'].isin(SECTION11_LNPS) & neighbor_cells['ova_positive']].copy()
ova_b_targets['lnp_call'] = ova_b_targets['lnp_call'].astype(str)
for marker in section11_required:
    ova_b_targets[marker] = pd.to_numeric(ova_b_targets[marker], errors='coerce')
ova_b_state_long = ova_b_targets.melt(id_vars=['source_obs_name', 'lnp_region', 'lnp_call'], value_vars=list(OVA_B_STATE_MARKERS), var_name='marker', value_name='expression').dropna(subset=['expression'])
ova_b_state_region = ova_b_state_long.groupby(['lnp_region', 'lnp_call', 'marker'], observed=True).agg(n_ova_positive_b=('source_obs_name', 'nunique'), mean_expression=('expression', 'mean'), median_expression=('expression', 'median')).reset_index()
ova_b_targets[['source_obs_name', 'lnp_region', 'lnp_call'] + list(OVA_B_STATE_MARKERS)].to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_state_per_cell.csv', index=False)
ova_b_state_region.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_state_by_region.csv', index=False)
ova_b_state_test_rows = []
for (marker, marker_label) in OVA_B_STATE_MARKERS.items():
    marker_data = ova_b_state_region[ova_b_state_region['marker'].eq(marker)].copy()
    for (metric, statistic_label) in [('mean_expression', 'Mean'), ('median_expression', 'Median')]:
        test = section11_paired_figure(marker_data, metric, f'{statistic_label} {marker_label} expression', f'OVA+ B-cell {marker_label}\n{statistic_label.lower()} expression', f'section11_lnp08_vs_lnp10_ova_positive_b_{marker.lower()}_{statistic_label.lower()}.png')
        test.update({'marker': marker, 'marker_label': marker_label, 'summary_statistic': statistic_label})
        ova_b_state_test_rows.append(test)
ova_b_state_tests = pd.DataFrame(ova_b_state_test_rows)
ova_b_state_tests.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_state_paired_ttests.csv', index=False)
ova_b_cd4_pairs = collect_target_cd4_neighbor_pairs(ova_b_targets, neighbor_cells, 'lnp_call', k=SECTION11_K, radius=SECTION11_RADIUS_PIXELS).rename(columns={'target_dc_index': 'target_b_index'})
ova_b_cd4_pairs['distance_um'] = pd.to_numeric(ova_b_cd4_pairs['distance'], errors='coerce') * 0.5
ova_b_cd4_target_metrics = ova_b_targets[['source_obs_name', 'lnp_region', 'lnp_call']].copy()
ova_b_cd4_target_metrics['target_b_index'] = ova_b_cd4_target_metrics.index
ova_b_cd4_pair_metrics = ova_b_cd4_pairs.groupby('target_b_index', observed=True).agg(n_cd4_neighbors=('cd4_index', 'nunique'), median_cd4_distance_um=('distance_um', 'median')).reset_index()
ova_b_cd4_target_metrics = ova_b_cd4_target_metrics.merge(ova_b_cd4_pair_metrics, on='target_b_index', how='left')
ova_b_cd4_target_metrics['n_cd4_neighbors'] = ova_b_cd4_target_metrics['n_cd4_neighbors'].fillna(0).astype(int)
ova_b_cd4_target_metrics['has_cd4_neighbor'] = ova_b_cd4_target_metrics['n_cd4_neighbors'] > 0
ova_b_cd4_spatial_region = ova_b_cd4_target_metrics.groupby(['lnp_region', 'lnp_call'], observed=True).agg(n_ova_positive_b=('target_b_index', 'nunique'), pct_with_cd4_neighbor=('has_cd4_neighbor', 'mean'), mean_n_cd4_neighbors=('n_cd4_neighbors', 'mean'), median_n_cd4_neighbors=('n_cd4_neighbors', 'median'), median_cd4_distance_um=('median_cd4_distance_um', 'median')).reset_index()
ova_b_cd4_spatial_region['pct_with_cd4_neighbor'] *= 100
ova_b_cd4_pairs.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_cd4_k10_radius25_neighbor_pairs.csv', index=False)
ova_b_cd4_target_metrics.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_cd4_k10_radius25_target_metrics.csv', index=False)
ova_b_cd4_spatial_region.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_cd4_k10_radius25_spatial_by_region.csv', index=False)
section11_spatial_specs = [('pct_with_cd4_neighbor', 'OVA+ B cells with ≥1 CD4+ T neighbor (%)', 'Nearby CD4+ T presence'), ('mean_n_cd4_neighbors', 'Mean CD4+ T count per OVA+ B cell', 'Mean nearby CD4+ T count'), ('median_n_cd4_neighbors', 'Median CD4+ T count per OVA+ B cell', 'Median nearby CD4+ T count'), ('median_cd4_distance_um', 'Median CD4+ T distance (µm)', 'Nearby CD4+ T distance')]
section11_spatial_test_rows = []
for (metric, ylabel, title) in section11_spatial_specs:
    test = section11_paired_figure(ova_b_cd4_spatial_region, metric, ylabel, f'{title}\nK=10, R<25 pixels (12.5 µm)', f'section11_lnp08_vs_lnp10_ova_positive_b_cd4_k10_radius25_{metric}.png')
    section11_spatial_test_rows.append(test)
section11_spatial_tests = pd.DataFrame(section11_spatial_test_rows)
section11_spatial_tests.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_cd4_k10_radius25_spatial_paired_ttests.csv', index=False)
section11_nearby_cd4 = ova_b_cd4_pairs.sort_values('distance').drop_duplicates(['lnp_region', 'lnp_call', 'cd4_index']).copy()
section11_cd4_values = neighbor_cells.loc[section11_nearby_cd4['cd4_index'], list(SECTION11_CD4_STATE_MARKERS)].apply(pd.to_numeric, errors='coerce').reset_index(drop=True)
section11_nearby_cd4 = pd.concat([section11_nearby_cd4.reset_index(drop=True), section11_cd4_values], axis=1)
section11_cd4_state_long = section11_nearby_cd4.melt(id_vars=['lnp_region', 'lnp_call', 'cd4_index', 'distance', 'distance_um'], value_vars=list(SECTION11_CD4_STATE_MARKERS), var_name='marker', value_name='expression').dropna(subset=['expression'])
section11_cd4_state_region = section11_cd4_state_long.groupby(['lnp_region', 'lnp_call', 'marker'], observed=True).agg(n_unique_cd4=('cd4_index', 'nunique'), mean_expression=('expression', 'mean'), median_expression=('expression', 'median')).reset_index()
section11_nearby_cd4.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_surrounding_unique_cd4.csv', index=False)
section11_cd4_state_region.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_surrounding_cd4_state_by_region.csv', index=False)
section11_cd4_state_test_rows = []
for (marker, marker_label) in SECTION11_CD4_STATE_MARKERS.items():
    marker_data = section11_cd4_state_region[section11_cd4_state_region['marker'].eq(marker)].copy()
    for (metric, statistic_label) in [('mean_expression', 'Mean'), ('median_expression', 'Median')]:
        test = section11_paired_figure(marker_data, metric, f'{statistic_label} {marker_label} expression', f'Surrounding CD4+ T-cell {marker_label}\n{statistic_label.lower()} expression; K=10, R<25 pixels', f'section11_lnp08_vs_lnp10_ova_positive_b_surrounding_cd4_{marker.lower()}_{statistic_label.lower()}.png')
        test.update({'marker': marker, 'marker_label': marker_label, 'summary_statistic': statistic_label})
        section11_cd4_state_test_rows.append(test)
section11_cd4_state_tests = pd.DataFrame(section11_cd4_state_test_rows)
section11_cd4_state_tests.to_csv(TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_surrounding_cd4_state_paired_ttests.csv', index=False)
display(ova_b_state_region)
display(ova_b_state_tests)
display(ova_b_cd4_spatial_region)
display(section11_spatial_tests)
display(section11_cd4_state_region)
display(section11_cd4_state_tests)


## 11. Supplementary Figure 11: neighborhoods of LNP-positive OVA-positive cells


In [ ]:
ova_lnp_cells = neighbor_cells.loc[neighbor_cells['lnp_positive'] & neighbor_cells['ova_positive'] & neighbor_cells['lnp_call'].isin(LNP_ORDER)].copy()
ova_lnp_cells['lnp_call'] = ova_lnp_cells['lnp_call'].astype(str)
ova_lnp_neighborhood_counts = ova_lnp_cells.groupby(['lnp_call', 'neighborhood'], observed=True).size().rename('n_ova_lnp_cells').reset_index()
ova_lnp_totals = ova_lnp_cells.groupby('lnp_call', observed=True).size().rename('n_ova_lnp_cells_total')
ova_lnp_neighborhood_counts = ova_lnp_neighborhood_counts.merge(ova_lnp_totals, on='lnp_call', how='left')
ova_lnp_neighborhood_counts['percent'] = 100 * ova_lnp_neighborhood_counts['n_ova_lnp_cells'] / ova_lnp_neighborhood_counts['n_ova_lnp_cells_total']
ova_lnp_neighborhood_counts.to_csv(TABLE_DIR / 'section13_corrected_lnp_ova_positive_neighborhood_abundance.csv', index=False)
ova_lnp_region_counts = ova_lnp_cells.groupby(['lnp_region', 'lnp_call', 'neighborhood'], observed=True).size().rename('n_ova_lnp_cells').reset_index()
ova_lnp_region_totals = ova_lnp_cells.groupby(['lnp_region', 'lnp_call'], observed=True).size().rename('n_ova_lnp_cells_total').reset_index()
ova_lnp_region_counts = ova_lnp_region_counts.merge(ova_lnp_region_totals, on=['lnp_region', 'lnp_call'], how='left')
ova_lnp_region_counts['percent'] = 100 * ova_lnp_region_counts['n_ova_lnp_cells'] / ova_lnp_region_counts['n_ova_lnp_cells_total']
ova_lnp_region_counts.to_csv(TABLE_DIR / 'section13_corrected_lnp_ova_positive_neighborhood_abundance_by_region.csv', index=False)
ova_lnp_plot = ova_lnp_neighborhood_counts.pivot(index='lnp_call', columns='neighborhood', values='percent').fillna(0).reindex(index=LNP_ORDER, columns=ordered_names, fill_value=0)
(fig, ax) = plt.subplots(figsize=(max(10, 0.65 * len(ordered_names)), 6))
sns.heatmap(ova_lnp_plot, cmap='Purples', annot=True, fmt='.1f', cbar_kws={'label': '% of decoded LNP+ OVA+ cells'}, ax=ax)
ax.set(title='Spatial neighborhood distribution of LNP+ OVA+ cells by corrected LNP identity', xlabel='Neighborhood', ylabel='Corrected LNP call')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()
section13_palette = dict(zip(ordered_names, sns.color_palette('tab20', n_colors=len(ordered_names))))
(fig, axes) = plt.subplots(2, 5, figsize=(20, 8.2), subplot_kw={'aspect': 'equal'})
for (ax, lnp_name) in zip(axes.flat, LNP_ORDER):
    values = ova_lnp_plot.loc[lnp_name, ordered_names].fillna(0).to_numpy(float)
    total = values.sum()
    n_ova_lnp_cells = int(ova_lnp_totals.get(lnp_name, 0))
    if total <= 0:
        ax.text(0.5, 0.5, 'No LNP+ OVA+ cells', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"{lnp_name.replace('_', ' ')}\n(n=0)", fontsize=12, fontweight='bold')
        ax.axis('off')
        continue
    ax.pie(values, colors=[section13_palette[name] for name in ordered_names], startangle=90, counterclock=False, autopct=lambda pct: f'{pct:.1f}%' if pct >= 4 else '', pctdistance=0.72, textprops={'fontsize': 8}, wedgeprops={'linewidth': 0.6, 'edgecolor': 'white'})
    ax.set_title(f"{lnp_name.replace('_', ' ')}\n(n={n_ova_lnp_cells:,})", fontsize=12, fontweight='bold')
section13_legend_handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=8, markerfacecolor=section13_palette[name], markeredgecolor='none', label=name) for name in ordered_names]
fig.legend(handles=section13_legend_handles, labels=ordered_names, title='Neighborhood', loc='center left', bbox_to_anchor=(0.82, 0.5), frameon=False, fontsize=9, title_fontsize=10)
fig.suptitle('Spatial neighborhood distribution of LNP+ OVA+ cells by corrected LNP identity', fontsize=17, y=0.98)
fig.subplots_adjust(left=0.03, right=0.8, bottom=0.04, top=0.9, wspace=0.08, hspace=0.24)
plt.show()
display(ova_lnp_neighborhood_counts)
display(ova_lnp_region_counts)


## 12. Final Figure 2 paired-comparison plotting code

These cells were copied from the later manuscript figure-generation notebooks. They read the region-level tables produced above and display the final plots inline.


### Mean CD86 paired LNP_08 versus LNP_10 boxplot

Boxplot with paired slide-region points/lines for mean CD86 expression in SIINFEKL-H-2Kb+ LNP+ DCs.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = CELL_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'lnp_positive_dc_siinfekl_ova_by_region.csv'
PDF_PATH = OUT / 'cd86_lnp08_vs_lnp10_mean_box_paired_nature.pdf'
PNG_PATH = OUT / 'cd86_lnp08_vs_lnp10_mean_box_paired_nature.png'
SOURCE_PATH = OUT / 'cd86_lnp08_vs_lnp10_mean_box_paired_source_data.csv'
TEST_PATH = OUT / 'cd86_lnp08_vs_lnp10_mean_box_paired_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
metric = 'mean_cd86_in_siinfekl_positive'
lnps = ['LNP_08', 'LNP_10']
df = pd.read_csv(INPUT_CSV)
wide = df[df['lnp_call'].isin(lnps)].pivot(index='lnp_region', columns='lnp_call', values=metric).reindex(columns=lnps).dropna().reset_index()
wide['region_id'] = wide['lnp_region'].str.extract('_(reg\\d+)$')[0]
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
paired_t = stats.ttest_rel(a, b)
wilcoxon = stats.wilcoxon(a, b, zero_method='wilcox', alternative='two-sided', method='auto')
test_df = pd.DataFrame([{'metric': metric, 'n_pairs': len(wide), 'lnp08_mean': float(np.mean(a)), 'lnp10_mean': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10': float(np.median(a - b)), 'paired_t_statistic': float(paired_t.statistic), 'paired_t_pvalue': float(paired_t.pvalue), 'wilcoxon_statistic': float(wilcoxon.statistic), 'wilcoxon_pvalue': float(wilcoxon.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
region_palette = {'reg000': '#2C7FB8', 'reg001': '#41AB5D', 'reg002': '#F28E2B', 'reg003': '#8E6BBE'}
replicate_labels = {'reg000': 'Bio. rep. 1', 'reg001': 'Bio. rep. 2', 'reg002': 'Bio. rep. 3', 'reg003': 'Bio. rep. 4'}
(fig, ax) = plt.subplots(figsize=(2.25, 2.95))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.0}, whiskerprops={'color': '#222222', 'linewidth': 0.75}, capprops={'color': '#222222', 'linewidth': 0.75}, boxprops={'edgecolor': '#222222', 'linewidth': 0.75})
for patch in box['boxes']:
    patch.set_facecolor('#7D9AB8')
    patch.set_alpha(0.48)
offsets = np.linspace(-0.12, 0.12, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    color = region_palette.get(row['region_id'], '#888888')
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    ax.plot([x0, x1], [y0, y1], color='#C9C9C9', linewidth=0.42, alpha=0.55, zorder=1)
    ax.scatter([x0, x1], [y0, y1], s=24, color=color, edgecolor='white', linewidth=0.65, alpha=0.96, zorder=5)
p_display = float(wilcoxon.pvalue)
ymax = max(np.nanmax(a), np.nanmax(b))
ymin = min(np.nanmin(a), np.nanmin(b))
yrange = ymax - ymin
yline = ymax + max(yrange * 0.09, 6)
br_h = max(yrange * 0.025, 2)
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + max(yrange * 0.035, 2), f'**  p={p_display:.4f}', ha='center', va='bottom', fontsize=7.0, color='#111111')
ax.set_title('CD86 in SIINFEKL-H-2Kb+ LNP+ DCs', fontsize=8.6, fontweight='bold', pad=7)
ax.set_ylabel('Mean CD86 expression')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=45, ha='right')
ax.set_xlim(-0.48, 1.48)
ax.set_ylim(max(0, ymin - max(yrange * 0.08, 5)), yline + br_h + max(yrange * 0.2, 12))
handles = [Line2D([0], [0], marker='o', linestyle='none', markersize=4.3, markerfacecolor=region_palette[r], markeredgecolor='white', markeredgewidth=0.45, label=replicate_labels[r]) for r in ['reg000', 'reg001', 'reg002', 'reg003']]
ax.legend(handles=handles, title='Biological replicate', title_fontsize=6.1, fontsize=5.8, loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0)
fig.text(0.12, 0.02, 'Lines connect paired slide-region measurements.', ha='left', va='bottom', fontsize=5.7, color='#555555')
fig.subplots_adjust(left=0.23, right=0.72, bottom=0.25, top=0.8)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### Median CD8+ T distance one-sided paired comparison

Complete-case paired comparison for the directional hypothesis that LNP_10 has greater median CD8+ T distance than LNP_08. Distances are converted to micrometers using 0.5 µm per coordinate unit.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_hybrid_settings_by_region.csv'
PDF_PATH = OUT / 'cd8_distance_lnp08_vs_lnp10_median_onesided_nature.pdf'
PNG_PATH = OUT / 'cd8_distance_lnp08_vs_lnp10_median_onesided_nature.png'
SOURCE_PATH = OUT / 'cd8_distance_lnp08_vs_lnp10_median_onesided_source_data.csv'
TEST_PATH = OUT / 'cd8_distance_lnp08_vs_lnp10_median_onesided_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
UM_PER_COORDINATE_UNIT = 0.5
raw_metric = 'median_cd8_distance'
metric = 'median_cd8_distance_um'
lnps = ['LNP_08', 'LNP_10']
raw = pd.read_csv(INPUT_CSV)
raw[metric] = raw[raw_metric] * UM_PER_COORDINATE_UNIT
df = raw[raw['radius'].eq(25) & raw['k_neighbors'].eq(10) & raw['lnp_call'].isin(lnps)].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values=metric).reindex(columns=lnps).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
coordinate_wide = df.pivot(index='lnp_region', columns='lnp_call', values=raw_metric).reindex(columns=lnps).dropna().reset_index()
coordinate_wide = coordinate_wide.rename(columns={lnp: f'{lnp}_coordinate_units' for lnp in lnps})
source_wide = wide.rename(columns={lnp: f'{lnp}_um' for lnp in lnps}).merge(coordinate_wide, on='lnp_region', how='left')
source_wide['um_per_coordinate_unit'] = UM_PER_COORDINATE_UNIT
source_wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
two_sided = stats.ttest_rel(a, b, alternative='two-sided')
one_sided = stats.ttest_rel(a, b, alternative='less')
test_df = pd.DataFrame([{'metric': metric, 'unit': 'um', 'um_per_coordinate_unit': UM_PER_COORDINATE_UNIT, 'hypothesis': 'LNP_10 median CD8+ T distance > LNP_08', 'test': 'one-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean_um': float(np.mean(a)), 'lnp10_mean_um': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10_um': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10_um': float(np.median(a - b)), 't_statistic': float(one_sided.statistic), 'one_sided_pvalue': float(one_sided.pvalue), 'two_sided_pvalue': float(two_sided.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#9DB8D0')
    patch.set_alpha(0.55)
base_offsets = np.linspace(-0.055, 0.055, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + base_offsets[pair_i]
    x1 = 1 + base_offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.38, alpha=0.7, zorder=1)
    ax.scatter([x0, x1], [y0, y1], s=21, color=color, edgecolor='white', linewidth=0.45, alpha=0.98, zorder=5)
p_display = float(one_sided.pvalue)
stars = '*' if p_display < 0.05 else 'ns'
ymax = max(np.nanmax(a), np.nanmax(b))
ymin = min(np.nanmin(a), np.nanmin(b))
yrange = ymax - ymin if ymax > ymin else 1
yline = ymax + max(yrange * 0.11, 0.325)
br_h = max(yrange * 0.03, 0.11)
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + max(yrange * 0.04, 0.14), f'{stars}  p={p_display:.3f}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('Median CD8+ T distance', fontsize=8.4, fontweight='bold', pad=7)
ax.set_ylabel('Median CD8+ T distance (µm)')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(ymin - max(yrange * 0.12, 0.375), yline + br_h + max(yrange * 0.18, 0.5))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.3, right=0.68, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### Nearby CD8+ T-cell presence one-sided paired comparison

Complete-case paired comparison of the percentage of SIINFEKL-H-2Kb+ LNP+ DCs with at least one nearby CD8+ T cell. Neighborhoods use K=10 with a radius <12.5 µm. The prespecified directional hypothesis is LNP_08 > LNP_10.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'lnp08_vs_lnp10_siinfekl_positive_dc_cd8_hybrid_settings_by_region.csv'
PDF_PATH = OUT / 'figure_2h_nearby_cd8_presence_onesided_revision.pdf'
PNG_PATH = OUT / 'figure_2h_nearby_cd8_presence_onesided_revision.png'
SOURCE_PATH = OUT / 'figure_2h_nearby_cd8_presence_onesided_revision_source_data.csv'
TEST_PATH = OUT / 'figure_2h_nearby_cd8_presence_onesided_revision_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
metric = 'pct_with_cd8_neighbor'
lnps = ['LNP_08', 'LNP_10']
raw = pd.read_csv(INPUT_CSV)
df = raw[raw['radius'].eq(25) & raw['k_neighbors'].eq(10) & raw['lnp_call'].isin(lnps)].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values=metric).reindex(columns=lnps).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
one_sided = stats.ttest_rel(a, b, alternative='greater')
two_sided = stats.ttest_rel(a, b, alternative='two-sided')
test_df = pd.DataFrame([{'metric': metric, 'hypothesis': 'LNP_08 > LNP_10', 'test': 'one-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean_percent': float(np.mean(a)), 'lnp10_mean_percent': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10_percent_points': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10_percent_points': float(np.median(a - b)), 't_statistic': float(one_sided.statistic), 'degrees_of_freedom': int(len(wide) - 1), 'one_sided_pvalue': float(one_sided.pvalue), 'two_sided_pvalue': float(two_sided.pvalue), 'k_neighbors': 10, 'radius_um': 12.5}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#9DB8D0')
    patch.set_alpha(0.55)
offsets = np.linspace(-0.055, 0.055, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.38, alpha=0.7, zorder=1)
    ax.scatter([x0, x1], [y0, y1], s=21, color=color, edgecolor='white', linewidth=0.45, alpha=0.98, zorder=5)
p_display = float(one_sided.pvalue)
stars = '*' if p_display < 0.05 else 'ns'
yline = 106
br_h = 2.5
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + 2.3, f'{stars}  p={p_display:.3f}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('Nearby CD8+ T-cell presence', fontsize=8.4, fontweight='bold', pad=12)
ax.text(0.5, 1.015, 'K=10; radius <12.5 µm', transform=ax.transAxes, ha='center', va='bottom', fontsize=6.4, color='#4C4C4C')
ax.set_ylabel('SIINFEKL-H-2Kb+ LNP+ DCs with\n≥1 nearby CD8+ T cell (%)')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(-4, 119)
ax.set_yticks(np.arange(0, 101, 20))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.34, right=0.68, bottom=0.22, top=0.80)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### Nearby CD4+ T-cell FOXP3 median expression

Paired region-level comparison for nearby CD4+ T cells around OVA+ LNP+ DCs.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'lnp08_vs_lnp10_ova_positive_dc_cd4_k10_radius25_activation_by_region.csv'
PDF_PATH = OUT / 'nearby_cd4_foxp3_median_lnp08_vs_lnp10_box_nature.pdf'
PNG_PATH = OUT / 'nearby_cd4_foxp3_median_lnp08_vs_lnp10_box_nature.png'
SOURCE_PATH = OUT / 'nearby_cd4_foxp3_median_lnp08_vs_lnp10_source_data.csv'
TEST_PATH = OUT / 'nearby_cd4_foxp3_median_lnp08_vs_lnp10_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
raw = pd.read_csv(INPUT_CSV)
df = raw[raw['lnp_call'].isin(['LNP_08', 'LNP_10']) & raw['marker'].eq('FOXP3')].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values='median_expression').reindex(columns=['LNP_08', 'LNP_10']).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
test = stats.ttest_rel(a, b, alternative='two-sided')
test_df = pd.DataFrame([{'metric': 'median_expression', 'marker': 'FOXP3', 'marker_label': 'FOXP3', 'test': 'two-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean': float(np.mean(a)), 'lnp10_mean': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10': float(np.median(a - b)), 't_statistic': float(test.statistic), 'pvalue': float(test.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#9DB8D0')
    patch.set_alpha(0.55)
offsets = np.linspace(-0.055, 0.055, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.38, alpha=0.7, zorder=1)
    ax.scatter([x0, x1], [y0, y1], s=21, color=color, edgecolor='white', linewidth=0.45, alpha=0.98, zorder=5)
p_display = float(test.pvalue)
stars = '**' if p_display < 0.01 else '*' if p_display < 0.05 else 'ns'
ymax = max(np.nanmax(a), np.nanmax(b))
ymin = min(np.nanmin(a), np.nanmin(b))
yrange = ymax - ymin if ymax > ymin else 1
yline = ymax + max(yrange * 0.11, 0.045)
br_h = max(yrange * 0.03, 0.015)
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + max(yrange * 0.04, 0.018), f'{stars}  p={p_display:.3f}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('Nearby CD4+ T-cell FOXP3', fontsize=8.4, fontweight='bold', pad=7)
ax.set_ylabel('Median FOXP3 expression')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(ymin - max(yrange * 0.12, 0.05), yline + br_h + max(yrange * 0.18, 0.06))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.28, right=0.68, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### OVA+ B-cell CD25 median expression

Paired region-level comparison for OVA+ LNP+ B cells assigned to LNP_08 or LNP_10.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_state_by_region.csv'
PDF_PATH = OUT / 'ova_b_cd19_median_lnp08_vs_lnp10_box_nature.pdf'
PNG_PATH = OUT / 'ova_b_cd19_median_lnp08_vs_lnp10_box_nature.png'
SOURCE_PATH = OUT / 'ova_b_cd19_median_lnp08_vs_lnp10_source_data.csv'
TEST_PATH = OUT / 'ova_b_cd19_median_lnp08_vs_lnp10_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
raw = pd.read_csv(INPUT_CSV)
df = raw[raw['lnp_call'].isin(['LNP_08', 'LNP_10']) & raw['marker'].eq('CD19')].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values='median_expression').reindex(columns=['LNP_08', 'LNP_10']).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
test = stats.ttest_rel(a, b, alternative='two-sided')
test_df = pd.DataFrame([{'metric': 'median_expression', 'marker': 'CD19', 'marker_label': 'CD19', 'test': 'two-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean': float(np.mean(a)), 'lnp10_mean': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10': float(np.median(a - b)), 't_statistic': float(test.statistic), 'pvalue': float(test.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#B8C96B')
    patch.set_alpha(0.5)
offsets = np.linspace(-0.065, 0.065, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.34, alpha=0.7, zorder=1, clip_on=True)
    ax.scatter([x0, x1], [y0, y1], s=19, color=color, edgecolor='white', linewidth=0.42, alpha=0.98, zorder=5, clip_on=True)
p_display = float(test.pvalue)
stars = '***' if p_display < 0.001 else '**' if p_display < 0.01 else '*' if p_display < 0.05 else 'ns'
ymax = max(np.nanmax(a), np.nanmax(b))
yline = ymax + 13
br_h = 4.5
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + 0.35, f'{stars}  p={p_display:.3g}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('OVA+ B-cell CD19', fontsize=8.4, fontweight='bold', pad=7)
ax.set_ylabel('Median CD19 expression')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(10, max(170, yline + br_h + 12))
ax.set_yticks(np.arange(20, 181, 20))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.28, right=0.68, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### OVA+ B-cell CD138 median expression

Paired region-level comparison for OVA+ LNP+ B cells assigned to LNP_08 or LNP_10.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_state_by_region.csv'
PDF_PATH = OUT / 'ova_b_cd138_median_lnp08_vs_lnp10_box_nature.pdf'
PNG_PATH = OUT / 'ova_b_cd138_median_lnp08_vs_lnp10_box_nature.png'
SOURCE_PATH = OUT / 'ova_b_cd138_median_lnp08_vs_lnp10_source_data.csv'
TEST_PATH = OUT / 'ova_b_cd138_median_lnp08_vs_lnp10_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
raw = pd.read_csv(INPUT_CSV)
df = raw[raw['lnp_call'].isin(['LNP_08', 'LNP_10']) & raw['marker'].eq('CD138')].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values='median_expression').reindex(columns=['LNP_08', 'LNP_10']).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
test = stats.ttest_rel(a, b, alternative='two-sided')
test_df = pd.DataFrame([{'metric': 'median_expression', 'marker': 'CD138', 'marker_label': 'CD138', 'test': 'two-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean': float(np.mean(a)), 'lnp10_mean': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10': float(np.median(a - b)), 't_statistic': float(test.statistic), 'pvalue': float(test.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#B8C96B')
    patch.set_alpha(0.5)
offsets = np.linspace(-0.065, 0.065, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.34, alpha=0.7, zorder=1, clip_on=True)
    ax.scatter([x0, x1], [y0, y1], s=19, color=color, edgecolor='white', linewidth=0.42, alpha=0.98, zorder=5, clip_on=True)
p_display = float(test.pvalue)
stars = '**' if p_display < 0.01 else '*' if p_display < 0.05 else 'ns'
yline = 4.05
br_h = 0.1
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + 0.12, f'{stars}  p={p_display:.3f}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('OVA+ B-cell CD138', fontsize=8.4, fontweight='bold', pad=7)
ax.set_ylabel('Median CD138 expression')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(0, 4.5)
ax.set_yticks(np.arange(0, 4.6, 1.0))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.28, right=0.68, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### OVA+ B-cell CD25 median expression

Paired region-level comparison for OVA+ LNP+ B cells assigned to LNP_08 or LNP_10.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_state_by_region.csv'
PDF_PATH = OUT / 'ova_b_cd25_median_lnp08_vs_lnp10_box_nature.pdf'
PNG_PATH = OUT / 'ova_b_cd25_median_lnp08_vs_lnp10_box_nature.png'
SOURCE_PATH = OUT / 'ova_b_cd25_median_lnp08_vs_lnp10_source_data.csv'
TEST_PATH = OUT / 'ova_b_cd25_median_lnp08_vs_lnp10_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
raw = pd.read_csv(INPUT_CSV)
df = raw[raw['lnp_call'].isin(['LNP_08', 'LNP_10']) & raw['marker'].eq('CD25')].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values='median_expression').reindex(columns=['LNP_08', 'LNP_10']).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
test = stats.ttest_rel(a, b, alternative='two-sided')
test_df = pd.DataFrame([{'metric': 'median_expression', 'marker': 'CD25', 'marker_label': 'CD25', 'test': 'two-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean': float(np.mean(a)), 'lnp10_mean': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10': float(np.median(a - b)), 't_statistic': float(test.statistic), 'pvalue': float(test.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#B8C96B')
    patch.set_alpha(0.5)
offsets = np.linspace(-0.065, 0.065, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.34, alpha=0.7, zorder=1, clip_on=True)
    ax.scatter([x0, x1], [y0, y1], s=19, color=color, edgecolor='white', linewidth=0.42, alpha=0.98, zorder=5, clip_on=True)
p_display = float(test.pvalue)
stars = '***' if p_display < 0.001 else '**' if p_display < 0.01 else '*' if p_display < 0.05 else 'ns'
yline = 36.2
br_h = 0.8
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + 0.35, f'{stars}  p={p_display:.3g}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('OVA+ B-cell CD25', fontsize=8.4, fontweight='bold', pad=7)
ax.set_ylabel('Median CD25 expression')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(5, 38)
ax.set_yticks(np.arange(5, 39, 5))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.28, right=0.68, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))


### Surrounding CD4+ T-cell FOXP3 median expression

Paired region-level comparison of CD4+ T cells surrounding OVA+ LNP+ B cells assigned to LNP_08 or LNP_10.

Final manuscript plotting code; the figure is displayed inline and is not saved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
PROJECT_DIR = NANOSTAMP_ROOT
TABLE_DIR = NEIGHBOR_TABLE_DIR
FIG_DIR = PROJECT_DIR / 'Manuscripts' / 'NanoSTAMP' / 'Figures'
OUT = GENERATED_ROOT / 'Figure_Source_Data'
OUT.mkdir(parents=True, exist_ok=True)
INPUT_CSV = TABLE_DIR / 'section11_lnp08_vs_lnp10_ova_positive_b_surrounding_cd4_state_by_region.csv'
PDF_PATH = OUT / 'ova_b_surrounding_cd4_foxp3_median_lnp08_vs_lnp10_box_nature.pdf'
PNG_PATH = OUT / 'ova_b_surrounding_cd4_foxp3_median_lnp08_vs_lnp10_box_nature.png'
SOURCE_PATH = OUT / 'ova_b_surrounding_cd4_foxp3_median_lnp08_vs_lnp10_source_data.csv'
TEST_PATH = OUT / 'ova_b_surrounding_cd4_foxp3_median_lnp08_vs_lnp10_test.csv'
mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans', 'Liberation Sans'], 'font.size': 7, 'axes.linewidth': 0.75, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'xtick.major.size': 2.6, 'ytick.major.size': 2.6, 'legend.frameon': False, 'savefig.dpi': 600})
raw = pd.read_csv(INPUT_CSV)
df = raw[raw['lnp_call'].isin(['LNP_08', 'LNP_10']) & raw['marker'].eq('FOXP3')].copy()
wide = df.pivot(index='lnp_region', columns='lnp_call', values='median_expression').reindex(columns=['LNP_08', 'LNP_10']).dropna().reset_index()
wide['bio_rep'] = wide['lnp_region'].str.extract('^(S\\d+)')[0].map({'S1': 'Bio. rep. 1', 'S2': 'Bio. rep. 2', 'S3': 'Bio. rep. 3', 'S4': 'Bio. rep. 4'})
wide.to_csv(SOURCE_PATH, index=False)
a = wide['LNP_08'].to_numpy(float)
b = wide['LNP_10'].to_numpy(float)
test = stats.ttest_rel(a, b, alternative='two-sided')
test_df = pd.DataFrame([{'metric': 'median_expression', 'marker': 'FOXP3', 'marker_label': 'FOXP3', 'test': 'two-sided paired Student t-test', 'n_pairs': len(wide), 'lnp08_mean': float(np.mean(a)), 'lnp10_mean': float(np.mean(b)), 'mean_paired_difference_lnp08_minus_lnp10': float(np.mean(a - b)), 'median_paired_difference_lnp08_minus_lnp10': float(np.median(a - b)), 't_statistic': float(test.statistic), 'pvalue': float(test.pvalue)}])
test_df.to_csv(TEST_PATH, index=False)
rep_colors = {'Bio. rep. 1': '#2C7FB8', 'Bio. rep. 2': '#41AB5D', 'Bio. rep. 3': '#F28E2B', 'Bio. rep. 4': '#8E63C7'}
(fig, ax) = plt.subplots(figsize=(2.15, 2.45))
box = ax.boxplot([a, b], positions=[0, 1], widths=0.46, patch_artist=True, showfliers=False, medianprops={'color': '#111111', 'linewidth': 1.05}, whiskerprops={'color': '#222222', 'linewidth': 0.8}, capprops={'color': '#222222', 'linewidth': 0.8}, boxprops={'edgecolor': '#222222', 'linewidth': 0.8})
for patch in box['boxes']:
    patch.set_facecolor('#B8C96B')
    patch.set_alpha(0.5)
offsets = np.linspace(-0.06, 0.06, len(wide))
for (pair_i, (_, row)) in enumerate(wide.iterrows()):
    x0 = 0 + offsets[pair_i]
    x1 = 1 + offsets[pair_i]
    y0 = row['LNP_08']
    y1 = row['LNP_10']
    color = rep_colors.get(row['bio_rep'], '#666666')
    ax.plot([x0, x1], [y0, y1], color='#B8B8B8', linewidth=0.36, alpha=0.7, zorder=1)
    ax.scatter([x0, x1], [y0, y1], s=20, color=color, edgecolor='white', linewidth=0.42, alpha=0.98, zorder=5)
p_display = float(test.pvalue)
stars = '**' if p_display < 0.01 else '*' if p_display < 0.05 else 'ns'
ymax = max(np.nanmax(a), np.nanmax(b))
ymin = min(np.nanmin(a), np.nanmin(b))
yrange = ymax - ymin if ymax > ymin else 1
yline = ymax + max(yrange * 0.11, 0.1)
br_h = max(yrange * 0.03, 0.035)
ax.plot([0, 0, 1, 1], [yline, yline + br_h, yline + br_h, yline], color='#111111', linewidth=0.75)
ax.text(0.5, yline + br_h + max(yrange * 0.04, 0.04), f'{stars}  p={p_display:.3f}', ha='center', va='bottom', fontsize=7.2)
ax.set_title('Surrounding CD4+ T-cell FOXP3', fontsize=8.4, fontweight='bold', pad=7)
ax.set_ylabel('Median FOXP3 expression')
ax.set_xlabel('')
ax.set_xticks([0, 1])
ax.set_xticklabels(['LNP 08', 'LNP 10'], rotation=42, ha='right')
ax.set_xlim(-0.42, 1.42)
ax.set_ylim(max(0, ymin - max(yrange * 0.12, 0.12)), yline + br_h + max(yrange * 0.18, 0.15))
ax.grid(False)
handles = [Line2D([0], [0], marker='o', linestyle='none', markerfacecolor=color, markeredgecolor='white', markeredgewidth=0.45, markersize=4.7, label=rep) for (rep, color) in rep_colors.items() if rep in set(wide['bio_rep'])]
ax.legend(handles=handles, title='Biological replicate', loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0, fontsize=6.2, title_fontsize=6.4, handletextpad=0.4, labelspacing=0.35)
fig.subplots_adjust(left=0.28, right=0.68, bottom=0.22, top=0.82)
plt.show()
print(f'Saved: {PDF_PATH}')
print(f'Saved: {PNG_PATH}')
print(f'Saved source data: {SOURCE_PATH}')
print(f'Saved test: {TEST_PATH}')
print(test_df.to_string(index=False))
